# Figure 6 | MSN spatial homogenization and disorganization

D1/D2 heatmaps, category profiles, gene CV distributions, and animal CV radar summaries.

Check input/output paths and settings before running each workflow. Run its cells in order with a fresh kernel. See README.md for required inputs, panel coverage, and configuration differences.

| Panels | Workflow |
|---|---|
| A | NAcc DV gene heatmaps |
| B-C | WT/Homo D1-versus-D2 category profiles |
| D | Genotype CV distributions |
| E | Five-axis CV radar |
| F | Schematic summary |


## Panel A | NAcc DV gene heatmaps

Select NAcc DV (`n-y`), D1/D2, and WT/Homo outputs. The default is relative expression, whereas the manuscript panel uses row Z scores. The plotting function also supports `row_zscore`.


In [ ]:
# Imports and settings.
from pathlib import Path
import re, gc, logging, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse


SOURCE_FILE = Path(r"F:\BASSFISH_HD\Source_data\1228genes_x_y_axis.h5ad")
OUT_ROOT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\1000gene") / "gene_axis_CV_aligned_relative_heatmaps"
OUT_ROOT.mkdir(parents=True, exist_ok=True)


REGION_AXIS_PAIRS = [("d", "x"), ("d", "y"), ("n", "x"), ("n", "y"), ("o", "x")]
TARGET_CELLTYPES = ["D1", "D2"]
TARGET_AXES = sorted({axis for _, axis in REGION_AXIS_PAIRS})
CONDITIONS = ["WT", "Heter", "Homo"]
BATCH_STOP_ON_ERROR = True


REGION_COL = "region_str_manual"
CELLTYPE_COL = "refined_major_cluster"
CONDITION_COL = "condition"
SAMPLE_COL = "sample"
AXIS_COLUMNS = {"x": "region_manual_axis_0_100", "y": "region_manual_axis_y_0_100"}
EXPRESSION_LAYER = "norm_1e4"
DETECTION_LAYER = "counts"

N_BINS = 10
R_THRESHOLD = 0.5
HEATMAP_AGGREGATION = "sample_mean"
PROFILE_NORMALIZATION = "gene_mean1"
ERROR_TYPE = "sem"


USE_WT_GENE_QC = True
MANUAL_GENES = None
QC_MIN_WT_MEAN_NORM = 1.0
QC_MIN_WT_DETECTION = 0.02
QC_MIN_WT_SAMPLE_DETECTION = 0.01
QC_MIN_WT_SAMPLE_FRACTION = 0.50
QC_MIN_WT_BIN_CELLS = 5
QC_MIN_VALID_WT_BINS = 8


HEATMAP_COLOR_MODE = "relative_mean"
RELATIVE_VMAX = 1.5
ZSCORE_VMAX = 0.5
CV_APPLY_VARIATION_QC = True
CV_MIN_VALID_BINS = 5
CV_MIN_BIN_MEAN_EXPR = 0.02
CV_MIN_EXPRESSED_BINS = 5
CV_MIN_PROFILE_MEAN_EXPR = 0.05
CV_MIN_SAMPLE_FRACTION_PER_CONDITION = 0.50
CV_ABSOLUTE_MAX = 1
CV_DELTA_MAX = 0.5
CV_COLOR_PERCENTILE = 99.0
RELATIVE_RESPECT_CV_QC = True
CMAP = "RdBu_r"
SHOW_GENE_NAMES = False
SHOW_COUNTS = False
PLOT_PER_SAMPLE = True
PLOT_COMBINED = True
SHOW_FIGURES = False
LINE_YLIM = None
DPI = 600
CONDITION_COLORS = {"WT":"#4cd964", "Heter":"#ff9500", "Homo":"#ff3b30"}
HEAT_PANEL_W_MM, HEAT_PANEL_H_MM = 12, 32
HEAT_PANEL_GAP_MM, HEAT_GROUP_GAP_MM = 0.5, 1
LINE_PANEL_W_MM, LINE_PANEL_H_MM = 28, 20

plt.rcParams.update({"font.family":"Arial", "font.size":7, "axes.labelsize":7,
                     "axes.titlesize":7, "xtick.labelsize":7, "ytick.labelsize":7,
                     "legend.fontsize":7, "pdf.fonttype":42, "ps.fonttype":42})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)

assert HEATMAP_AGGREGATION in {"sample_mean","cell_mean"}
assert PROFILE_NORMALIZATION in {"gene_mean1","none"}
assert ERROR_TYPE in {"sem","sd"}
assert HEATMAP_COLOR_MODE in {"relative_mean", "row_zscore"}
assert N_BINS == 10, "This analysis requires ten equal-width bins over 0-100"
print("Input:", SOURCE_FILE)
print("Output:", OUT_ROOT)
TASKS = [(reg, ct, ax) for reg, ax in REGION_AXIS_PAIRS for ct in TARGET_CELLTYPES]
if len(TASKS) != len(set(TASKS)):
    raise ValueError("REGION_AXIS_PAIRS / TARGET_CELLTYPES contains duplicate tasks")
print(f"Batch plan: {len(TASKS)} targets -> {TASKS}")


In [ ]:
if not SOURCE_FILE.exists():
    raise FileNotFoundError(f"Data file not found: {SOURCE_FILE}. Check the full filename and path.")
adata = sc.read_h5ad(SOURCE_FILE)
print(adata)
if EXPRESSION_LAYER not in adata.layers:
    raise KeyError(f"Required layers[{EXPRESSION_LAYER!r}] is missing; available layers: {list(adata.layers.keys())}. Fallback to potentially scaled adata.X is disabled.")
for col in [REGION_COL, CELLTYPE_COL, CONDITION_COL, SAMPLE_COL, *[AXIS_COLUMNS[k] for k in TARGET_AXES]]:
    if col not in adata.obs:
        raise KeyError(f"Input obs is missing {col!r}; available columns: {list(adata.obs.columns)}")
if not adata.var_names.is_unique:
    raise ValueError("var_names contains duplicate genes; resolve them first")

X = adata.layers[EXPRESSION_LAYER]
if sparse.issparse(X):

    if len(X.data) and np.nanmin(X.data) < 0:
        raise ValueError("norm_1e4 contains negative values and cannot be used as normalized expression")
else:
    if np.nanmin(X) < 0:
        raise ValueError("norm_1e4 contains negative values and cannot be used as normalized expression")

if DETECTION_LAYER in adata.layers:
    DETECTION_MATRIX = adata.layers[DETECTION_LAYER]
    detection_origin = f"layers[{DETECTION_LAYER}]"
else:
    DETECTION_MATRIX = X
    detection_origin = f"layers[{EXPRESSION_LAYER}] > 0"
    print("Note: no counts layer; gene detection is defined as norm_1e4 > 0")

obs = adata.obs
region_all = obs[REGION_COL].astype(str).to_numpy()
ctype_all = obs[CELLTYPE_COL].astype(str).to_numpy()
condition_all = obs[CONDITION_COL].astype(str).to_numpy()
sample_all = obs[SAMPLE_COL].astype(str).to_numpy()
axis_all = {k: pd.to_numeric(obs[v], errors="coerce").to_numpy(float) for k,v in AXIS_COLUMNS.items() if k in TARGET_AXES}
genes_all = np.asarray(adata.var_names.astype(str))
print("Conditions:", pd.Series(condition_all).value_counts().to_dict())
print("Cell types preview:", sorted(pd.unique(ctype_all))[:35])


In [ ]:
BIN_CENTERS = np.arange(N_BINS) * (100 / N_BINS) + (50 / N_BINS)
BIN_LABELS = [f"{i*10}–{(i+1)*10}" for i in range(N_BINS)]
CLASS_ORDER = ["high_to_low", "non_gradient", "low_to_high"]
CLASS_TITLES = {"high_to_low":"High to low", "non_gradient":"Non-gradient", "low_to_high":"Low to high"}

def safe_nanmean(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(a, axis=axis)

def safe_nanstd(a, axis=None, ddof=0):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanstd(a, axis=axis, ddof=ddof)

def mean_rows(mat, rows):
    if not len(rows):
        return np.full(mat.shape[1], np.nan)
    a = mat[rows]
    return np.asarray(a.mean(axis=0)).ravel() if sparse.issparse(a) else safe_nanmean(np.asarray(a), axis=0)

def detect_fraction(mat, rows):
    if not len(rows):
        return np.zeros(mat.shape[1], float)
    a = mat[rows]
    return np.asarray((a > 0).sum(axis=0)).ravel() / len(rows)

def row_zscore(matrix):
    matrix = np.asarray(matrix, float)
    mu = safe_nanmean(matrix, axis=1)[:,None]
    sd = safe_nanstd(matrix, axis=1)[:,None]

    result = np.full(matrix.shape, np.nan, dtype=float)
    valid = np.isfinite(matrix) & np.isfinite(sd) & (sd > 0)
    np.divide(matrix - mu, sd, out=result, where=valid)
    flat = (np.isfinite(sd[:,0]) & (sd[:,0] == 0))
    if flat.any():
        result[flat] = np.where(np.isfinite(matrix[flat]), 0., np.nan)
    return result


def relative_to_own_mean(matrix):
    """Compute bin expression divided by its spatial mean, minus one."""
    matrix = np.asarray(matrix, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        mu = np.nanmean(matrix, axis=1, keepdims=True)
    result = np.full_like(matrix, np.nan, dtype=float)
    np.divide(matrix, mu, out=result,
              where=np.isfinite(matrix) & np.isfinite(mu) & (mu>0))
    result[np.isfinite(result)] -= 1.0
    result[np.isfinite(matrix).sum(axis=1)<CV_MIN_VALID_BINS,:] = np.nan
    return result


def sample_gene_cv(matrix):
    """Compute population SD divided by the positive spatial mean."""
    a=np.asarray(matrix, float)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)
        mu=np.nanmean(a, axis=1)
        sd=np.nanstd(a, axis=1, ddof=0)
    good=(np.isfinite(a).sum(axis=1)>=CV_MIN_VALID_BINS) & np.isfinite(mu) & (mu>0)
    out=np.full(a.shape[0],np.nan,float)
    out[good]=sd[good]/mu[good]
    return out


def variation_sample_qc_mask(matrix):
    """Apply per-sample expression QC criteria."""
    a=np.asarray(matrix,float)
    valid=np.isfinite(a).sum(axis=1)>=CV_MIN_VALID_BINS
    if not CV_APPLY_VARIATION_QC:
        return valid
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning)
        mu=np.nanmean(a,axis=1)
    expressed=np.sum(np.isfinite(a)&(a>=CV_MIN_BIN_MEAN_EXPR),axis=1)
    return valid & (mu>=CV_MIN_PROFILE_MEAN_EXPR) & (expressed>=CV_MIN_EXPRESSED_BINS)

def wt_pearson_rows(matrix):
    result = np.zeros(matrix.shape[0], float)
    for i,y in enumerate(matrix):
        good = np.isfinite(y)
        if good.sum() >= 3 and np.nanstd(y[good]) > 0:
            result[i] = np.corrcoef(BIN_CENTERS[good], y[good])[0,1]
    return result

def select_gene_mask(target_gene_names, Xsubset, Dsubset, cond, sample, bins, wt_matrix):
    selected = np.ones(len(target_gene_names), bool)
    if MANUAL_GENES is not None:
        wanted = list(dict.fromkeys(str(x) for x in MANUAL_GENES))
        missing = sorted(set(wanted) - set(target_gene_names))
        if missing: raise KeyError(f"Manually selected genes are absent from the panel: {missing}")
        selected &= np.isin(target_gene_names, wanted)
    audit = pd.DataFrame({"gene": target_gene_names, "manual_pass": selected})
    wt_rows = np.flatnonzero(cond == "WT")
    qc = {}
    if USE_WT_GENE_QC:
        if len(wt_rows) == 0: raise ValueError("WT gene QC requires WT cells")
        wt_mean = mean_rows(Xsubset, wt_rows)
        wt_detect = detect_fraction(Dsubset, wt_rows)
        wt_samples = sorted(pd.unique(sample[wt_rows]))
        sf = np.stack([detect_fraction(Dsubset, wt_rows[sample[wt_rows] == s]) for s in wt_samples])
        frac_samples = (sf >= QC_MIN_WT_SAMPLE_DETECTION).mean(axis=0)
        bvalid = np.zeros(len(target_gene_names), int)
        for b in range(N_BINS):
            br = wt_rows[bins[wt_rows] == b]
            if len(br) >= QC_MIN_WT_BIN_CELLS:
                bvalid += (detect_fraction(Dsubset, br) > 0).astype(int)
        qc_pass = ((wt_mean >= QC_MIN_WT_MEAN_NORM) &
                   (wt_detect >= QC_MIN_WT_DETECTION) &
                   (frac_samples >= QC_MIN_WT_SAMPLE_FRACTION) &
                   (bvalid >= QC_MIN_VALID_WT_BINS))
        selected &= qc_pass
        qc = {"WT_mean_norm_1e4":wt_mean, "WT_detection_fraction":wt_detect,
              "WT_fraction_samples_ge_1pct":frac_samples,
              "WT_valid_expressed_bins":bvalid, "WT_QC_pass":qc_pass}
        for k,v in qc.items(): audit[k] = v
    audit["selected"] = selected
    if not selected.any(): raise ValueError("No genes passed filtering; check the criteria")
    return selected, audit


In [ ]:
def plot_heat_panels(panels, sorted_names, counts_by_key, out_pdf, axis_label, page_title):

    if not panels: return
    left, right, bottom, top, cb_gap, cb_w = 16, 5, 17, 16, 4, 3
    xs = []; cursor = left; prev_group = None
    for title, group, _ in panels:
        if prev_group is not None and group != prev_group:
            cursor += HEAT_GROUP_GAP_MM - HEAT_PANEL_GAP_MM
        xs.append(cursor)
        cursor += HEAT_PANEL_W_MM + HEAT_PANEL_GAP_MM
        prev_group = group
    heat_right = xs[-1] + HEAT_PANEL_W_MM
    fw = heat_right + cb_gap + cb_w + right
    fh = bottom + HEAT_PANEL_H_MM + top
    fig = plt.figure(figsize=(fw/25.4, fh/25.4))
    try:

        fig.text(left/fw, (bottom + HEAT_PANEL_H_MM + 11)/fh, page_title,
                 ha="left", va="center", fontsize=8)
        im = None
        for i, ((title, group, z), xp) in enumerate(zip(panels, xs)):
            ax = fig.add_axes([xp/fw, bottom/fh, HEAT_PANEL_W_MM/fw, HEAT_PANEL_H_MM/fh])
            im = ax.imshow(np.ma.masked_invalid(z), cmap=CMAP, vmin=-ZSCORE_VMAX,
                           vmax=ZSCORE_VMAX, aspect="auto", interpolation="nearest",
                           rasterized=False)
            ax.set_title(title, pad=2)
            count = counts_by_key[(title,group)]
            labels = ([f"{b+1}\n{int(count[b])}" for b in range(N_BINS)] if SHOW_COUNTS
                      else [str(b+1) for b in range(N_BINS)])
            ax.set_xticks(np.arange(N_BINS)); ax.set_xticklabels(labels, fontsize=6)
            ax.set_xlabel(axis_label, fontsize=7)
            if i == 0 and SHOW_GENE_NAMES and len(sorted_names) <= 65:
                ax.set_yticks(np.arange(len(sorted_names)))
                ax.set_yticklabels(sorted_names, fontsize=5)
            else:
                ax.set_yticks([])
            for s in ax.spines.values(): s.set_linewidth(.5)
            ax.tick_params(length=2, width=.5, pad=1)
        if im is not None:
            cax = fig.add_axes([(heat_right+cb_gap)/fw, bottom/fh, cb_w/fw, HEAT_PANEL_H_MM/fh])
            cb = fig.colorbar(im,cax=cax)
            cb.set_label("Row z-score")
            cb.ax.tick_params(width=.5, length=2, pad=1)
            cb.outline.set_linewidth(.5)
        fig.savefig(out_pdf, dpi=DPI, bbox_inches="tight", pad_inches=.03, facecolor="white")
        if SHOW_FIGURES: plt.show()
    finally:
        plt.close(fig)
    print("Saved",out_pdf)

def plot_class_profiles(summary, category_counts, out_pdf, axis_label, page_title):
    left, right, bottom, top, gap = 13, 6, 14, 15, 7
    fw = left + 3*LINE_PANEL_W_MM + 2*gap + right
    fh = bottom + LINE_PANEL_H_MM + top
    fig = plt.figure(figsize=(fw/25.4, fh/25.4))
    try:
        fig.text(left/fw, (bottom + LINE_PANEL_H_MM + 10)/fh, page_title,
                 ha="left", va="center", fontsize=8)
        axes=[]
        for j,cat in enumerate(CLASS_ORDER):
            xp=left + j*(LINE_PANEL_W_MM + gap)
            ax=fig.add_axes([xp/fw, bottom/fh, LINE_PANEL_W_MM/fw, LINE_PANEL_H_MM/fh])
            axes.append(ax)
            for cond in CONDITIONS:
                sub = summary[(summary["category"]==cat)&(summary["condition"]==cond)].sort_values("bin")
                if sub.empty: continue
                x = sub["bin_center"].to_numpy(float)
                y = sub["mean"].to_numpy(float)
                e = sub[ERROR_TYPE].to_numpy(float)
                ax.plot(x,y,lw=1.0,label=cond,color=CONDITION_COLORS[cond])
                g=np.isfinite(e) & np.isfinite(y)
                if g.any():
                    ax.fill_between(x,y-e,y+e,where=g,color=CONDITION_COLORS[cond],alpha=.18,lw=0)
            if PROFILE_NORMALIZATION == "gene_mean1":
                ax.axhline(1,color="0.7",lw=.5,ls="--")
            ax.set_xlim(0,100);ax.set_xticks([0,20,40,60,80,100]);ax.set_xlabel(axis_label)
            if LINE_YLIM is not None: ax.set_ylim(*LINE_YLIM)
            ax.set_title(f"{CLASS_TITLES[cat]} (n={category_counts.get(cat,0)})",pad=2)
            for s in ax.spines.values():s.set_linewidth(.5)
            ax.tick_params(width=.5,length=2,pad=1)
        axes[0].set_ylabel("Relative expression (gene mean=1)" if PROFILE_NORMALIZATION == "gene_mean1"
                           else "Mean expression (norm_1e4)")
        axes[-1].legend(frameon=False,fontsize=6,handlelength=1.4,loc="best")
        fig.savefig(out_pdf,dpi=DPI,bbox_inches="tight",pad_inches=.03,facecolor="white")
        if SHOW_FIGURES:plt.show()
    finally:
        plt.close(fig)
    print("Saved",out_pdf)


In [ ]:
def cv_heatmap_plot(prefix, names, cv_by_condition, delta_by_condition, outpath):

    n=len(names)
    absolute=np.column_stack([cv_by_condition[c] for c in CONDITIONS])
    diffs=np.column_stack([delta_by_condition['Heter'],delta_by_condition['Homo']])
    finite_cv=absolute[np.isfinite(absolute)]
    finite_d=diffs[np.isfinite(diffs)]
    vmax=(CV_ABSOLUTE_MAX if CV_ABSOLUTE_MAX is not None else
          max(.1, float(np.percentile(finite_cv,CV_COLOR_PERCENTILE))) if len(finite_cv) else 1.)
    dmax=(CV_DELTA_MAX if CV_DELTA_MAX is not None else
          max(.05, float(np.percentile(np.abs(finite_d),CV_COLOR_PERCENTILE))) if len(finite_d) else .5)

    fig=plt.figure(figsize=(4.35, max(2.1,min(8., n*.015+1.2))))
    try:
        ax1=fig.add_axes([.10,.11,.34,.79]); ax2=fig.add_axes([.56,.11,.22,.79])
        cmap1=plt.get_cmap('Reds').copy();cmap1.set_bad('white')
        cmap2=plt.get_cmap('RdBu_r').copy();cmap2.set_bad('white')
        im1=ax1.imshow(np.ma.masked_invalid(absolute),aspect='auto',interpolation='nearest',
                       cmap=cmap1,vmin=0,vmax=vmax,rasterized=False)
        im2=ax2.imshow(np.ma.masked_invalid(diffs),aspect='auto',interpolation='nearest',
                       cmap=cmap2,vmin=-dmax,vmax=dmax,rasterized=False)
        ax1.set_xticks(np.arange(3),CONDITIONS);ax2.set_xticks([0,1],['Heter−WT','Homo−WT'])
        for ax in (ax1,ax2):
            ax.set_yticks([]);ax.tick_params(axis='x',length=0,labelrotation=30,labelsize=6)
            for sp in ax.spines.values():sp.set_linewidth(.4)
        if SHOW_GENE_NAMES and n<=65:
            ax1.set_yticks(range(n));ax1.set_yticklabels(names,fontsize=4.5)
        fig.text(.10,.955,f'{prefix.replace("_"," | ")}  |  per-gene CV (animal mean) and ΔCV',fontsize=8,va='center')
        ca1=fig.add_axes([.455,.11,.022,.79]);ca2=fig.add_axes([.79,.11,.022,.79])
        cb1=fig.colorbar(im1,cax=ca1);cb1.set_label('CV',fontsize=7)
        cb2=fig.colorbar(im2,cax=ca2);cb2.set_label('ΔCV',fontsize=7)
        cb1.ax.tick_params(labelsize=6,length=2);cb2.ax.tick_params(labelsize=6,length=2)
        fig.savefig(outpath,dpi=DPI,bbox_inches='tight',pad_inches=.03,facecolor='white')
    finally:
        plt.close(fig)
    print('Saved',outpath)


def cv_by_animal_for_target(prefix,samples_raw,meta,sample_info,outdir):

    genes=meta['gene'].to_numpy(str)
    records=[]
    cv_array={}
    pass_array={}
    for c,s in sample_info:
        mat=samples_raw[(c,s)]
        val=sample_gene_cv(mat)
        passes=variation_sample_qc_mask(mat)
        val[~passes]=np.nan
        cv_array[(c,s)]=val
        pass_array[(c,s)]=passes
        for i,g in enumerate(genes):
            records.append({'condition':c,'sample':s,'gene':g,'cv':val[i],
                            'pass_variation_sample_QC':bool(passes[i]),
                            'n_valid_bins':int(np.isfinite(mat[i]).sum()),
                            'profile_mean':float(safe_nanmean(mat[i]))})
    df=pd.DataFrame(records)
    df.to_csv(outdir/f'{prefix}_12_gene_sample_CV_matched_variation.csv',index=False)
    cv_by_c={}; count_by_c={}; fraction_by_c={}
    crows=[]
    for c in CONDITIONS:
        keys=[k for k in sample_info if k[0]==c]
        values=np.stack([cv_array[k] for k in keys],axis=0)
        nfinite=np.isfinite(values).sum(axis=0)

        req=(max(1,int(np.ceil(len(keys)*CV_MIN_SAMPLE_FRACTION_PER_CONDITION)))
             if CV_APPLY_VARIATION_QC else 1)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore',RuntimeWarning)
            avg=np.nanmean(values,axis=0)
            sd=np.nanstd(values,axis=0,ddof=1) if len(keys)>1 else np.full(values.shape[1],np.nan)
        avg[nfinite<req]=np.nan
        cv_by_c[c]=avg;count_by_c[c]=nfinite
        fraction_by_c[c]=nfinite/len(keys)
        for i,g in enumerate(genes):
            crows.append({'gene':g,'condition':c,'cv_animal_mean':avg[i],
                          'n_animals_valid':nfinite[i], 'n_animals_total':len(keys),
                          'min_animals_required':req,'cv_sd_animals':sd[i],
                          'cv_sem_animals':sd[i]/np.sqrt(nfinite[i]) if nfinite[i]>1 else np.nan})
    paired=np.isfinite(cv_by_c['WT']) & np.isfinite(cv_by_c['Heter']) & np.isfinite(cv_by_c['Homo'])
    differences={'Heter':np.where(paired,cv_by_c['Heter']-cv_by_c['WT'],np.nan),
                 'Homo':np.where(paired,cv_by_c['Homo']-cv_by_c['WT'],np.nan)}
    pd.DataFrame(crows).to_csv(outdir/f'{prefix}_13_gene_condition_CV_matched_variation.csv',index=False)
    rows=[]
    for i,g in enumerate(genes):
        rows.append({'gene':g,'spatial_category':meta['spatial_category'].iat[i],
                     'WT_CV':cv_by_c['WT'][i], 'Heter_CV':cv_by_c['Heter'][i],
                     'Homo_CV':cv_by_c['Homo'][i],
                     'Heter_minus_WT':differences['Heter'][i],
                     'Homo_minus_WT':differences['Homo'][i],
                     'paired_all_conditions':bool(paired[i])})
    pd.DataFrame(rows).to_csv(outdir/f'{prefix}_14_gene_CV_signed_delta_WT.csv',index=False)
    if np.any(paired):
        print(prefix,'CV paired genes:',int(paired.sum()),
              'median WT/Heter/Homo:',*[round(float(np.nanmedian(cv_by_c[c][paired])),4) for c in CONDITIONS])
    else:
        print(prefix,'WARNING: no paired genes passed CV sample/condition QC')
    return cv_by_c,differences


In [ ]:
def analyze_one(region_target, celltype_target, axis_name):
    prefix=f"{region_target}_{celltype_target}_{axis_name}"
    od = OUT_ROOT
    page_prefix = f"{celltype_target} | {region_target}-{axis_name} axis"
    axis_values=axis_all[axis_name]
    pattern=rf"(?:^|_){re.escape(celltype_target)}(?:$|_)"
    ct_match=pd.Series(ctype_all).str.contains(pattern,regex=True,na=False).to_numpy(bool)
    valid = (region_all == region_target) & ct_match & np.isfinite(axis_values)
    valid &= (axis_values >= 0)&(axis_values <=100)& np.isin(condition_all,CONDITIONS)
    valid &= (sample_all != "nan") & (sample_all != "<NA>") & (sample_all != "")
    idx=np.flatnonzero(valid)
    if len(idx)==0:
        raise ValueError(f"{prefix}: no matching cells. Check region, celltype, and axis: {sorted(pd.unique(ctype_all))[:40]}")
    co=condition_all[idx]; sa=sample_all[idx]; a=axis_values[idx]
    bi=np.minimum(np.floor(a / (100/N_BINS)).astype(int),N_BINS-1)

    Xsub=X[idx]
    Dsub=DETECTION_MATRIX[idx] if USE_WT_GENE_QC else None
    sample_data={}; sample_counts={}; sample_info=[]; bin_qc=[]
    for c in CONDITIONS:
        ss=sorted(pd.unique(sa[co==c]))
        if not ss: raise ValueError(f"{prefix}: group {c} has no cells; the complete three-condition plot cannot be drawn")
        for s in ss:
            si=np.flatnonzero((co==c)&(sa==s))
            profile=np.full((adata.n_vars,N_BINS),np.nan)
            count=np.zeros(N_BINS,dtype=int)
            for b in range(N_BINS):
                ri=si[bi[si]==b]
                count[b]=len(ri)
                if len(ri): profile[:,b]=mean_rows(Xsub,ri)
                bin_qc.append({"region":region_target,"celltype":celltype_target,
                               "axis":axis_name,"condition":c,"sample":s,"bin":b+1,
                               "axis_left":b*10,"axis_right":(b+1)*10,"n_cells":len(ri)})
            sample_info.append((c,s))
            sample_data[(c,s)] = profile
            sample_counts[(c,s)] = count
    pooled_data={}; pooled_counts={}
    for c in CONDITIONS:
        keys=[(cc,s) for cc,s in sample_info if cc==c]
        if HEATMAP_AGGREGATION=="sample_mean":
            pooled_data[c] = safe_nanmean(np.stack([sample_data[k] for k in keys]),axis=0)
        else:
            pooled_data[c] = np.full((adata.n_vars,N_BINS),np.nan)
            for b in range(N_BINS):
                rows=np.flatnonzero((co==c)&(bi==b))
                if len(rows): pooled_data[c][:,b] = mean_rows(Xsub,rows)
        pooled_counts[c]=np.sum([sample_counts[k] for k in keys],axis=0)

    selected,gene_qc = select_gene_mask(genes_all, Xsub, Dsub, co, sa, bi, pooled_data["WT"])
    gene_qc.to_csv(od / f"{prefix}_00_gene_filter_QC.csv",index=False)
    selected_genes=genes_all[selected]
    nsel=len(selected_genes)
    wt_r=wt_pearson_rows(pooled_data["WT"][selected])
    cats=np.full(nsel,"non_gradient",dtype=object)
    cats[wt_r <= -R_THRESHOLD]="high_to_low"
    cats[wt_r >= R_THRESHOLD]="low_to_high"
    meta=pd.DataFrame({"gene":selected_genes,"WT_axis_r":wt_r,"spatial_category":cats,
                       "pattern_strength":np.abs(wt_r)})
    meta=meta.sort_values(["WT_axis_r","gene"],ascending=[True,True]).reset_index(drop=True)
    order=pd.Index(selected_genes).get_indexer(meta["gene"])
    meta.insert(0,"rank",np.arange(1,nsel+1))
    meta.to_csv(od / f"{prefix}_01_WT_fixed_gene_order_and_classes.csv",index=False)

    pooled_raw={c:pooled_data[c][selected][order] for c in CONDITIONS}
    pooled_z={c:row_zscore(pooled_raw[c]) for c in CONDITIONS}
    samples_raw={(c,s):sample_data[(c,s)][selected][order] for c,s in sample_info}
    samples_z={(c,s):row_zscore(samples_raw[(c,s)]) for c,s in sample_info}

    samples_rel={}
    for c,s in sample_info:
        arr=relative_to_own_mean(samples_raw[(c,s)])
        if RELATIVE_RESPECT_CV_QC:
            arr[~variation_sample_qc_mask(samples_raw[(c,s)]),:]=np.nan
        samples_rel[(c,s)]=arr
    pooled_rel={}
    for c in CONDITIONS:
        subset=[samples_rel[(cc,ss)] for cc,ss in sample_info if cc==c]

        pooled_rel[c]=safe_nanmean(np.stack(subset),axis=0)
        if RELATIVE_RESPECT_CV_QC:
            ok=np.sum(np.isfinite(np.stack(subset)).any(axis=2),axis=0)
            minimum=(max(1,int(np.ceil(len(subset)*CV_MIN_SAMPLE_FRACTION_PER_CONDITION)))
                     if CV_APPLY_VARIATION_QC else 1)
            pooled_rel[c][ok<minimum,:]=np.nan
    if HEATMAP_COLOR_MODE == 'relative_mean':
        plot_pooled,plot_samples=pooled_rel,samples_rel
    else:
        plot_pooled,plot_samples=pooled_z,samples_z

    cv_condition,cv_delta=cv_by_animal_for_target(prefix,samples_raw,meta,sample_info,od)
    cv_heatmap_plot(prefix,meta['gene'].to_list(),cv_condition,cv_delta,
                    od/f'{prefix}_15_CV_gene_condition_and_signed_change_heatmap.pdf')

    record_raw=[];record_z=[]
    for c in CONDITIONS:
        for i,g in enumerate(meta["gene"]):
            rec={"condition":c,"gene":g}
            record_raw.append({**rec,**{f"bin_{b+1}":pooled_raw[c][i,b] for b in range(N_BINS)}})
            record_z.append({**rec,**{f"bin_{b+1}":pooled_z[c][i,b] for b in range(N_BINS)}})
    pd.DataFrame(record_raw).to_csv(od / f"{prefix}_02_condition_bin_mean_expression.csv",index=False)
    pd.DataFrame(record_z).to_csv(od / f"{prefix}_03_condition_bin_row_zscore.csv",index=False)

    rel_condition=[]; rel_sample=[]
    for c in CONDITIONS:
        for i,g in enumerate(meta['gene']):
            rel_condition.append({'condition':c,'gene':g,
                                  **{f'bin_{b+1}':pooled_rel[c][i,b] for b in range(N_BINS)}})
    for c,ss in sample_info:
        for i,g in enumerate(meta['gene']):
            rel_sample.append({'condition':c,'sample':ss,'gene':g,
                               **{f'bin_{b+1}':samples_rel[(c,ss)][i,b] for b in range(N_BINS)}})
    pd.DataFrame(rel_condition).to_csv(od/f'{prefix}_16_condition_relative_to_sample_mean.csv',index=False)
    pd.DataFrame(rel_sample).to_csv(od/f'{prefix}_17_sample_relative_to_own_mean.csv',index=False)


    record_samples=[]
    for c,s in sample_info:
        for i,g in enumerate(meta["gene"]):
            record_samples.append({"condition":c,"sample":s,"gene":g,
                                   **{f"bin_{b+1}":samples_raw[(c,s)][i,b] for b in range(N_BINS)}})
    pd.DataFrame(record_samples).to_csv(od / f"{prefix}_04_sample_gene_bin_mean_expression.csv",index=False)
    pd.DataFrame(bin_qc).to_csv(od / f"{prefix}_05_sample_bin_cell_counts.csv",index=False)


    class_records=[]
    classes=meta["spatial_category"].to_numpy()
    for c,s in sample_info:
        mat=samples_raw[(c,s)]
        if PROFILE_NORMALIZATION=="gene_mean1":
            gene_mean=safe_nanmean(mat,axis=1)[:,None]
            norm=np.full_like(mat,np.nan,dtype=float)
            np.divide(mat,gene_mean,out=norm,where=np.isfinite(mat)&np.isfinite(gene_mean)&(gene_mean>0))
        else:
            norm=mat.copy()
        for cat in CLASS_ORDER:
            gm=classes==cat
            if not gm.any(): continue
            m=safe_nanmean(norm[gm],axis=0)
            for b,v in enumerate(m):
                class_records.append({"condition":c,"sample":s,"category":cat,"bin":b+1,
                                      "bin_center":BIN_CENTERS[b],"value":v,
                                      "n_class_genes":int(gm.sum())})
    class_df=pd.DataFrame(class_records)
    class_df.to_csv(od / f"{prefix}_06_sample_class_profiles.csv",index=False)
    class_summary=class_df.groupby(["condition","category","bin","bin_center"],observed=True)["value"].agg(["mean","std","count"]).reset_index()
    class_summary["sem"] = class_summary["std"] / np.sqrt(class_summary["count"])
    class_summary.to_csv(od / f"{prefix}_07_condition_class_mean_sem.csv",index=False)


    names=meta["gene"].tolist()
    cond_panels=[(c,c,plot_pooled[c]) for c in CONDITIONS]
    cond_map={(c,c):pooled_counts[c] for c in CONDITIONS}
    sample_panels=[(f"{s}\n{c}",c,plot_samples[(c,s)]) for c,s in sample_info]
    sample_map={(f"{s}\n{c}",c):sample_counts[(c,s)] for c,s in sample_info}
    axis_title=f"{axis_name} axis (0–100)"
    plot_heat_panels(cond_panels,names,cond_map,od / f"{prefix}_08_WT_Heter_Homo_heatmap.pdf",axis_title,
                     f"{page_prefix} | WT / Heter / Homo pooled heatmap")
    if PLOT_PER_SAMPLE:
        plot_heat_panels(sample_panels,names,sample_map,od / f"{prefix}_09_all_samples_heatmap.pdf",axis_title,
                         f"{page_prefix} | per-sample heatmap")
    if PLOT_COMBINED:
        group_panels=[(f"Pooled {c}","Pooled",plot_pooled[c]) for c in CONDITIONS]
        group_map={(f"Pooled {c}","Pooled"):pooled_counts[c] for c in CONDITIONS}
        plot_heat_panels(group_panels+sample_panels,names,{**group_map,**sample_map},
                         od / f"{prefix}_10_condition_plus_samples_heatmap.pdf",axis_title,
                         f"{page_prefix} | pooled conditions + per-sample heatmap")
    category_counts=meta["spatial_category"].value_counts().to_dict()
    plot_class_profiles(class_summary,category_counts,od / f"{prefix}_11_WT_gene_class_profiles_mean_sem.pdf",axis_title,
                        f"{page_prefix} | WT-defined gene classes | mean ± {ERROR_TYPE.upper()}")
    print(prefix,"n_cells",len(idx),"n_genes",nsel,"categories",category_counts)
    del Xsub, Dsub, sample_data, samples_raw, samples_z, samples_rel, pooled_rel
    gc.collect()
    return {"region":region_target,"celltype":celltype_target,"axis":axis_name,"n_cells":len(idx),
            "n_genes":nsel,"n_WT_samples":len(set(sa[co=="WT"])),
            "n_Heter_samples":len(set(sa[co=="Heter"])),"n_Homo_samples":len(set(sa[co=="Homo"])),
            "expression_source":EXPRESSION_LAYER,"QC_enabled":USE_WT_GENE_QC,
            "output":str(od)}


In [ ]:
results=[]
for job_idx, (region, ct, axis) in enumerate(TASKS, 1):
    print(f"\n{'='*65}\n[{job_idx}/{len(TASKS)}] {region} / {ct} / {axis}")
    try:
        info = analyze_one(region, ct, axis)
        results.append({**info, "status":"saved", "error":""})
    except Exception as exc:
        import traceback
        print(f"ERROR [{region}_{ct}_{axis}]: {type(exc).__name__}: {exc}")
        traceback.print_exc()
        results.append({"region":region,"celltype":ct,"axis":axis,
                        "status":"error","error":f"{type(exc).__name__}: {exc}"})
        if BATCH_STOP_ON_ERROR:
            pd.DataFrame(results).to_csv(OUT_ROOT/"_analysis_summary.csv",index=False)
            raise

    pd.DataFrame(results).to_csv(OUT_ROOT/"_analysis_summary.csv",index=False)
summary = pd.DataFrame(results)
print("\nCompleted:", OUT_ROOT)
print(summary[["region","celltype","axis","status"]].to_string(index=False))
print(f"Saved {int((summary['status']=='saved').sum())}/{len(TASKS)} combinations.")


## Panels B-C | D1-versus-D2 category profiles

NAcc DV profile outputs are named `per_celltype_n_y_WT_01_D1_D2_class_profiles.pdf` and the corresponding Homo file.


In [ ]:
# Imports and settings.
from pathlib import Path
import re, gc, warnings, logging
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import sparse


SOURCE_FILE = Path(r"F:\BASSFISH_HD\Source_data\1228genes_x_y_axis.h5ad")
OUT_ROOT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\1000gene") / "D1_D2_5axes_gradient_slope_dispersion_comparison"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

AXIS_SPECS = [("d", "x"), ("d", "y"), ("n", "x"), ("n", "y"), ("o", "x")]
TYPES = ["D1", "D2"]
CONDITIONS = ["WT", "Heter", "Homo"]
CATEGORIES = ["high_to_low", "non_gradient", "low_to_high"]
CLASS_NAMES = {"high_to_low":"High-to-low", "non_gradient":"NG", "low_to_high":"Low-to-high"}
CLASS_STYLES = {"high_to_low":"-", "non_gradient":":", "low_to_high":"--"}
TYPE_COLORS = {"D1":"#2471A3", "D2":"#D97706"}
AXIS_MARKERS = {("d","x"):"o", ("d","y"):"s", ("n","x"):"^", ("n","y"):"D", ("o","x"):"P"}

REGION_COL = "region_str_manual"
CELLTYPE_COL = "refined_major_cluster"
COND_COL = "condition"
SAMPLE_COL = "sample"
AXIS_COLS = {"x":"region_manual_axis_0_100", "y":"region_manual_axis_y_0_100"}
EXPR_LAYER = "norm_1e4"
DETECTION_LAYER = "counts"


GENE_SET_MODE = "per_celltype"
USE_WT_GENE_QC = True
MANUAL_GENES = None
N_BINS, R_THRESHOLD = 10, 0.5
PROFILE_NORMALIZATION = "gene_mean1"
QC_MIN_WT_MEAN_NORM = 1.0
QC_MIN_WT_DETECTION = 0.02
QC_MIN_WT_SAMPLE_DETECTION = 0.01
QC_MIN_WT_SAMPLE_FRACTION = 0.50
QC_MIN_WT_BIN_CELLS = 5
QC_MIN_VALID_WT_BINS = 8


MIN_VALID_POINTS_FIT = 8
SHOW_SEM_RIBBON = True
Y_LIMITS = (0.5, 1.5)
DPI = 600
FONT_PT = 7
SHOW_FIGURES = False
PLOT_RESIDUAL_RMSE = False

assert GENE_SET_MODE in {"per_celltype", "shared_consensus"}
assert PROFILE_NORMALIZATION == "gene_mean1", "gene_mean1 normalization is required for D1/D2 comparison"
assert len(AXIS_SPECS) == 5 and N_BINS == 10
plt.rcParams.update({"font.family":"Arial", "font.size":FONT_PT, "axes.labelsize":FONT_PT,
    "axes.titlesize":FONT_PT, "xtick.labelsize":FONT_PT, "ytick.labelsize":FONT_PT,
    "legend.fontsize":FONT_PT, "pdf.fonttype":42, "ps.fonttype":42})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)
print("Input:", SOURCE_FILE, "\nOutput:", OUT_ROOT)


In [ ]:
if not SOURCE_FILE.is_file():
    raise FileNotFoundError(f"Data file not found: {SOURCE_FILE}")
ad = sc.read_h5ad(SOURCE_FILE)
needed = [REGION_COL, CELLTYPE_COL, COND_COL, SAMPLE_COL] + list(AXIS_COLS.values())
missing = [c for c in needed if c not in ad.obs]
if missing:
    raise KeyError(f"obs is missing fields: {missing}")
if EXPR_LAYER not in ad.layers:
    raise KeyError(f"Required layers[{EXPR_LAYER!r}] is missing; available: {list(ad.layers.keys())}; fallback to adata.X is disabled")
if not ad.var_names.is_unique:
    raise ValueError("Duplicate gene names prevent unambiguous matching")
X = ad.layers[EXPR_LAYER]
DET = ad.layers[DETECTION_LAYER] if DETECTION_LAYER in ad.layers else X
genes = np.asarray(ad.var_names.astype(str))
obs = ad.obs
conds = obs[COND_COL].astype("string").fillna("").astype(str).to_numpy()
samples = obs[SAMPLE_COL].astype("string").fillna("").astype(str).to_numpy()
regions = obs[REGION_COL].astype("string").fillna("").astype(str).to_numpy()
celltypes = obs[CELLTYPE_COL].astype("string").fillna("").astype(str).to_numpy()
coords = {k:pd.to_numeric(obs[v],errors="coerce").to_numpy(float) for k,v in AXIS_COLS.items()}
CENTERS = np.arange(5,100,10,dtype=float)
if MANUAL_GENES is not None:
    missing_genes = sorted(set(MANUAL_GENES)-set(genes))
    if missing_genes: raise KeyError(f"Manually selected genes are missing: {missing_genes}")
print(ad)
print("Genes:", len(genes), "QC:", USE_WT_GENE_QC, "mode:", GENE_SET_MODE)


In [ ]:
def nmean(array,axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore",RuntimeWarning)
        return np.nanmean(array,axis=axis)

def mean_rows(mat,ix):
    if len(ix)==0: return np.full(mat.shape[1],np.nan)
    v=mat[ix]
    return np.asarray(v.mean(0)).ravel() if sparse.issparse(v) else nmean(np.asarray(v),axis=0)

def detect_rows(mat,ix):
    if not len(ix):return np.zeros(mat.shape[1],float)
    v=mat[ix]
    return (np.asarray((v>0).sum(0)).ravel()/len(ix) if sparse.issparse(v)
            else np.mean(np.asarray(v)>0,axis=0))

def pearson_rowwise(mat):
    out=np.zeros(mat.shape[0],float)
    for i,y in enumerate(mat):
        ok=np.isfinite(y)
        if np.sum(ok)>=3 and np.nanstd(y[ok])>0:
            out[i]=np.corrcoef(CENTERS[ok],y[ok])[0,1]
    return out

def classification(r):
    c=np.full(len(r),"non_gradient",dtype=object)
    c[r<=-R_THRESHOLD]="high_to_low"
    c[r>=R_THRESHOLD]="low_to_high"
    return c

def select_wt_genes(global_cells, local_bins, local_conds, local_samples):
    picked=np.ones(len(genes),bool)
    if MANUAL_GENES is not None:
        picked &= np.isin(genes,MANUAL_GENES)
    details={"manual_selected":np.isin(genes,MANUAL_GENES) if MANUAL_GENES is not None else np.ones(len(genes),bool)}
    if USE_WT_GENE_QC:
        wt_local=np.flatnonzero(local_conds=="WT")
        if not len(wt_local):raise ValueError("No WT cells")
        wt_global=global_cells[wt_local]
        means=mean_rows(X,wt_global)
        detection=detect_rows(DET,wt_global)
        wt_samples=np.unique(local_samples[wt_local])
        sample_det=np.stack([detect_rows(DET,wt_global[local_samples[wt_local]==s]) for s in wt_samples])
        fraction=np.mean(sample_det>=QC_MIN_WT_SAMPLE_DETECTION,axis=0)
        valid_bin=np.zeros(len(genes),int)
        for b in range(N_BINS):
            ix=wt_global[local_bins[wt_local]==b]
            if len(ix)>=QC_MIN_WT_BIN_CELLS:
                valid_bin+=(detect_rows(DET,ix)>0).astype(int)
        passed=((means>=QC_MIN_WT_MEAN_NORM)&(detection>=QC_MIN_WT_DETECTION)&
                (fraction>=QC_MIN_WT_SAMPLE_FRACTION)&(valid_bin>=QC_MIN_VALID_WT_BINS))
        picked &= passed
        details.update(WT_mean_norm=means,WT_detect=detection,
                       WT_fraction_samples_detect=fraction,WT_detected_bins=valid_bin,WT_pass=passed)
    details["selected"]=picked
    return picked,details

def fit_10point_curve(y):
    """Fit slope on coordinates divided by 100; calculate SD and residual RMSE."""
    y=np.asarray(y,float)
    good=np.isfinite(y)
    n=int(good.sum())
    result={"n_bins":n,"slope_100":np.nan,"intercept":np.nan,
            "std10":np.nan,"residual_rmse":np.nan,"r_squared":np.nan,"range10":np.nan}
    if n<MIN_VALID_POINTS_FIT:return result
    x=CENTERS[good]/100.0
    yy=y[good]
    slope,intercept=np.polyfit(x,yy,1)
    pred=slope*x+intercept
    mse=np.mean((yy-pred)**2)
    ss=np.sum((yy-np.mean(yy))**2)
    result.update(slope_100=float(slope),intercept=float(intercept),
        std10=float(np.std(yy,ddof=0)), residual_rmse=float(np.sqrt(mse)),
        range10=float(np.max(yy)-np.min(yy)),
        r_squared=float(1-np.sum((yy-pred)**2)/ss) if ss>1e-14 else np.nan)
    return result


In [ ]:
def build_base(region,axis_name,celltype):
    v=coords[axis_name]

    pat=rf"(?:^|_){re.escape(celltype)}(?:$|_)"
    cellmask=pd.Series(celltypes).str.contains(pat,regex=True,na=False).to_numpy(bool)
    chosen=np.flatnonzero((regions==region)&cellmask&np.isfinite(v)&
        (v>=0)&(v<=100)&np.isin(conds,CONDITIONS)&(samples!=""))
    if not len(chosen): raise ValueError(f"{region}-{axis_name} {celltype}: no matching cells")
    local_cond=conds[chosen]; local_sample=samples[chosen]
    bins=np.minimum(np.floor(v[chosen]/10).astype(int),9)
    sample_raw={}; cell_counts=[]
    for c in CONDITIONS:
        for s in sorted(pd.unique(local_sample[local_cond==c])):
            local_idx=np.flatnonzero((local_cond==c)&(local_sample==s))
            mat=np.full((len(genes),N_BINS),np.nan,float)
            for b in range(N_BINS):
                subset=local_idx[bins[local_idx]==b]
                if len(subset):mat[:,b]=mean_rows(X,chosen[subset])
                cell_counts.append({"region":region,"axis":axis_name,"celltype":celltype,
                    "condition":c,"sample":s,"bin":b+1,"n_cells":len(subset)})
            sample_raw[(c,s)]=mat
    wt=[mat for (c,s),mat in sample_raw.items() if c=="WT"]
    if not wt:raise ValueError(f"{region}-{axis_name} {celltype}: no WT animals")
    wt_mean=nmean(np.stack(wt),axis=0)
    r=pearson_rowwise(wt_mean)
    categories=classification(r)
    selected,audit=select_wt_genes(chosen,bins,local_cond,local_sample)
    audit_df=pd.DataFrame({"region":region,"axis":axis_name,"celltype":celltype,
        "gene":genes,"WT_r":r,"WT_category":categories,**audit})
    sample_norm={}
    for key,mat in sample_raw.items():
        m=nmean(mat,axis=1)[:,None]
        sample_norm[key]=np.divide(mat,m,out=np.full_like(mat,np.nan),where=np.isfinite(m)&(m>0))
    return {"region":region,"axis":axis_name,"celltype":celltype,
        "sample_norm":sample_norm,"r":r,"categories":categories,"selected":selected,
        "audit":audit_df,"bin_counts":pd.DataFrame(cell_counts)}

base={}
for region,axis in AXIS_SPECS:
    for celltype in TYPES:
        print("Prepare",region,axis,celltype)
        base[(region,axis,celltype)]=build_base(region,axis,celltype)
qc=pd.concat([v["audit"] for v in base.values()],ignore_index=True)
qc.to_csv(OUT_ROOT/"00_WT_gene_class_and_QC_all_axes.csv",index=False)
pd.concat([v["bin_counts"] for v in base.values()],ignore_index=True).to_csv(
    OUT_ROOT/"00_sample_bin_cell_counts_all_axes.csv",index=False)
print("Prepared 10 target analyses")


In [ ]:
def gene_mask_for(region,axis,celltype,category):
    own=base[(region,axis,celltype)]
    good=own["selected"]&(own["categories"]==category)
    if GENE_SET_MODE=="shared_consensus":
        other_type="D2" if celltype=="D1" else "D1"
        other=base[(region,axis,other_type)]
        good &= other["selected"]&(other["categories"]==category)
    return good

sample_rows=[]; count_rows=[]
for region,axis in AXIS_SPECS:
    for category in CATEGORIES:
        d1mask=gene_mask_for(region,axis,"D1",category)
        d2mask=gene_mask_for(region,axis,"D2",category)
        count_rows.append({"region":region,"axis":axis,"category":category,
            "mode":GENE_SET_MODE,"D1_genes":int(d1mask.sum()),"D2_genes":int(d2mask.sum()),
            "same_gene_set":bool(np.array_equal(d1mask,d2mask))})
    for celltype in TYPES:
        dat=base[(region,axis,celltype)]
        for category in CATEGORIES:
            mask=gene_mask_for(region,axis,celltype,category)
            n=int(mask.sum())
            for (condition,sample),mat in dat["sample_norm"].items():
                curve=nmean(mat[mask],axis=0) if n else np.full(N_BINS,np.nan)
                for b in range(N_BINS):
                    sample_rows.append({"region":region,"axis":axis,"celltype":celltype,
                        "condition":condition,"sample":sample,"category":category,
                        "bin":b+1,"axis_center":CENTERS[b],"value":curve[b],"n_genes":n})
count_df=pd.DataFrame(count_rows)
count_df.to_csv(OUT_ROOT/f"01_class_gene_count_{GENE_SET_MODE}.csv",index=False)
print(count_df.to_string(index=False))
if (count_df[["D1_genes","D2_genes"]].to_numpy()==0).any():
    warnings.warn("At least one spatial axis/class has no eligible genes; corresponding curves and fits will be empty. See 01_class_gene_count.csv")
sp=pd.DataFrame(sample_rows)
sp.to_csv(OUT_ROOT/f"02_animal_class_10bin_profiles_{GENE_SET_MODE}.csv",index=False)


animal_fits=[]
for key,g in sp.groupby(["region","axis","celltype","condition","sample","category"],sort=False):
    rec=dict(zip(["region","axis","celltype","condition","sample","category"],key))
    rec.update(fit_10point_curve(g.sort_values("bin")["value"].to_numpy(float)))
    animal_fits.append(rec)
animal_fits=pd.DataFrame(animal_fits)
animal_fits.to_csv(OUT_ROOT/f"03_animal_curve_slope_dispersion_{GENE_SET_MODE}.csv",index=False)


summary=sp.groupby(["region","axis","celltype","condition","category","bin","axis_center"],sort=False)["value"].agg(
    mean="mean",sd="std",n="count").reset_index()
summary["sem"]=summary["sd"]/np.sqrt(summary["n"])
summary.to_csv(OUT_ROOT/f"04_condition_class_10bin_mean_sem_{GENE_SET_MODE}.csv",index=False)


condition_fits=[]
for key,g in summary.groupby(["region","axis","celltype","condition","category"],sort=False):
    rec=dict(zip(["region","axis","celltype","condition","category"],key))
    rec.update(fit_10point_curve(g.sort_values("bin")["mean"].to_numpy(float)))
    matched=animal_fits
    sub=matched.loc[np.logical_and.reduce([(matched[k]==rec[k]) for k in ["region","axis","celltype","condition","category"]])]
    rec["n_animals_fit"]=int(sub["slope_100"].notna().sum())
    rec["animal_mean_slope_100"]=sub["slope_100"].mean()
    rec["animal_mean_std10"]=sub["std10"].mean()
    rec["animal_mean_residual_rmse"]=sub["residual_rmse"].mean()
    condition_fits.append(rec)
condition_fits=pd.DataFrame(condition_fits)
condition_fits.to_csv(OUT_ROOT/f"05_condition_mean_line_slope_dispersion_{GENE_SET_MODE}.csv",index=False)


k=["region","axis","condition","sample","category"]
paired=animal_fits.loc[animal_fits.celltype=="D1",k+["slope_100","std10","residual_rmse"]].merge(
    animal_fits.loc[animal_fits.celltype=="D2",k+["slope_100","std10","residual_rmse"]],
    on=k,how="inner",suffixes=("_D1","_D2"),validate="one_to_one")
for m in ["slope_100","std10","residual_rmse"]:
    paired[m+"_D2_minus_D1"]=paired[m+"_D2"]-paired[m+"_D1"]
paired.to_csv(OUT_ROOT/f"06_paired_D1_D2_animal_differences_{GENE_SET_MODE}.csv",index=False)
print("Animal profiles:",len(sp),"condition fits:",len(condition_fits),"paired animals:",len(paired))


In [ ]:
def plot_axis_condition(region,axis,condition):
    fig,ax=plt.subplots(figsize=(100/25.4,75/25.4))
    for celltype in TYPES:
        for category in CATEGORIES:
            g=summary.loc[(summary.region==region)&(summary.axis==axis)&
                (summary.celltype==celltype)&(summary.condition==condition)&
                (summary.category==category)].sort_values("bin")
            if g.empty:continue
            x=g.axis_center.to_numpy(float)
            y=g["mean"].to_numpy(float)
            if not np.isfinite(y).any():continue
            color=TYPE_COLORS[celltype]
            label=f"{celltype} | {CLASS_NAMES[category]}"
            ax.plot(x,y,color=color,ls=CLASS_STYLES[category],lw=1.25,label=label)
            if SHOW_SEM_RIBBON:
                sem=g["sem"].fillna(0).to_numpy(float)
                ax.fill_between(x,y-sem,y+sem,color=color,alpha=.085,lw=0)
    ax.axhline(1,color="0.6",lw=.6,ls=":")
    ax.set_xlim(0,100);ax.set_xticks([0,20,40,60,80,100])
    if Y_LIMITS is not None:ax.set_ylim(*Y_LIMITS)
    ax.set_xlabel(f"{region}-{axis} axis (0–100)")
    ax.set_ylabel("Relative class expression (gene mean = 1)")
    ax.set_title(f"MSN-D1 vs MSN-D2  |  {region}-{axis}  |  {condition}",pad=6)
    ax.legend(frameon=False,ncol=2,fontsize=6,loc="best")
    ax.tick_params(width=.5,length=2)
    for spine in ax.spines.values():spine.set_linewidth(.5)
    fig.tight_layout(pad=.7)
    path=OUT_ROOT/f"{GENE_SET_MODE}_{region}_{axis}_{condition}_01_D1_D2_class_profiles.pdf"
    try:
        fig.savefig(path,dpi=DPI,transparent=False,facecolor="white")
        if SHOW_FIGURES:plt.show()
    finally:plt.close(fig)
    return path

profile_pdfs=[]
for region,axis in AXIS_SPECS:
    for condition in CONDITIONS:
        p=plot_axis_condition(region,axis,condition)
        profile_pdfs.append(p)
print("Saved profile PDFs:",len(profile_pdfs))


## Panel D | Per-gene anatomical CV distributions

Also calculates RDR and exports CSV inputs for the radar workflow. The batch includes OT DV (`o-y`) in addition to the five manuscript axes. CV requires at least five valid bins. Values above 2 are clipped in the plot; exported values and statistics are unaffected.


In [ ]:
# Imports and settings.
from pathlib import Path
import gc, math, re, logging, warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import wilcoxon, mannwhitneyu


SOURCE_FILE = Path(r"F:\BASSFISH_HD\Source_data\1228genes_x_y_axis.h5ad")
OUTPUT_ROOT = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\1000gene") / "gene_axis_overall_variation_original_QC_optional_RDR_CV_FLAT"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def output_file(prefix, filename):
    return OUTPUT_ROOT / f"{prefix}_{filename}"


TARGET_REGIONS = ["d", "n", "o"]
TARGET_CELLTYPES = ["D1", "D2"]
TARGET_AXES = ["x", "y"]
METRICS = ["relative_dynamic_range", "cv"]
CONDITIONS = ["WT", "Heter", "Homo"]

REGION_COL = "region_str_manual"
CELLTYPE_COL = "refined_major_cluster"
COND_COL = "condition"
SAMPLE_COL = "sample"
AXIS_COLUMNS = {"x": "region_manual_axis_0_100", "y": "region_manual_axis_y_0_100"}
EXPR_LAYER = "norm_1e4"
N_BINS = 10
MIN_VALID_BINS_FOR_VARIATION = 5


USE_GENE_QC = True
MANUAL_GENES = None
MIN_BIN_MEAN_EXPR = 0.02
MIN_EXPRESSED_BINS = 5
MIN_PROFILE_MEAN_EXPR = 0.05
MIN_VALID_SAMPLE_FRACTION_PER_CONDITION = 0.50


DPI = 600
CMAP_COLORS = {"WT": "#4cd964", "Heter": "#ff9500", "Homo": "#ff3b30"}
SHOW_PAIRED_LINES = False
SHOW_FIGURES = False
DRAW_ANIMAL_LEVEL = True

CV_SCATTER_YLIM = (0.0, 2.0)

PLOT_Y_MIN, PLOT_Y_MAX = -0.1, 2

SHOW_NS_SIGNIFICANCE = True
AXES_WIDTH_MM, AXES_HEIGHT_MM = 25, 32
LEFT_MM, RIGHT_MM, BOTTOM_MM, TOP_MM = 13, 6, 12, 14

plt.rcParams.update({
    "font.family": "Arial", "font.size": 7, "axes.labelsize": 7,
    "axes.titlesize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7,
    "legend.fontsize": 7, "pdf.fonttype": 42, "ps.fonttype": 42,
})
logging.getLogger("fontTools.subset").setLevel(logging.ERROR)

if not SOURCE_FILE.is_file():
    raise FileNotFoundError(f"Input file not found: {SOURCE_FILE}")
if not METRICS or any(m not in {"relative_dynamic_range", "cv", "dynamic_range"} for m in METRICS):
    raise ValueError(f"Invalid METRICS setting: {METRICS}")
print("Source:", SOURCE_FILE)
print("Output:", OUTPUT_ROOT)
print("Expression QC:", USE_GENE_QC)


In [ ]:
def mean_rows(Xmat, rows):
    if len(rows) == 0:
        return np.full(Xmat.shape[1], np.nan)
    z = Xmat[rows]
    if sparse.issparse(z):
        return np.asarray(z.mean(axis=0)).ravel()
    return np.nanmean(np.asarray(z), axis=0)


def calc_variation(profile, metric):
    x = np.asarray(profile, float)
    x = x[np.isfinite(x)]
    if len(x) < MIN_VALID_BINS_FOR_VARIATION:
        return np.nan
    mean = np.nanmean(x)
    dr = np.nanmax(x) - np.nanmin(x)
    if metric == "relative_dynamic_range":
        return dr / mean if mean > 0 else np.nan
    if metric == "cv":
        return np.nanstd(x, ddof=0) / mean if mean > 0 else np.nan
    if metric == "dynamic_range":
        return dr
    raise ValueError(metric)


def make_sample_profiles(adata, axis_name):
    xcol = AXIS_COLUMNS[axis_name]
    X = adata.layers[EXPR_LAYER]
    axis = pd.to_numeric(adata.obs[xcol], errors="coerce").to_numpy(float)
    valid_axis = np.isfinite(axis) & (axis >= 0) & (axis <= 100)
    bins = np.full(len(axis), -1, dtype=int)
    bins[valid_axis] = np.clip(np.floor(axis[valid_axis] / 10), 0, N_BINS-1).astype(int)
    cond = adata.obs[COND_COL].astype(str).to_numpy()
    sample = adata.obs[SAMPLE_COL].astype(str).to_numpy()
    result, bin_qc = {}, []
    for c in CONDITIONS:
        samples = sorted(pd.unique(sample[valid_axis & (cond == c)]))
        print(f"  {axis_name} {c} samples: {samples}")
        for s in samples:
            sm = valid_axis & (cond == c) & (sample == s)
            mat = np.full((adata.n_vars, N_BINS), np.nan, float)
            for b in range(N_BINS):
                rows = np.where(sm & (bins == b))[0]
                mat[:,b] = mean_rows(X, rows)
                bin_qc.append({"condition":c, "sample":s, "bin":b+1,
                               "left":b*10, "right":(b+1)*10, "n_cells":len(rows)})
            result[(c,s)] = mat
    if not result:
        raise ValueError("No valid axis samples matched WT/Heter/Homo")
    return result, pd.DataFrame(bin_qc)


def gene_profile_qc(sample_profiles, genes, prefix):

    qc_records = []
    for (c,s), mat in sample_profiles.items():
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', RuntimeWarning)
            profile_mean = np.nanmean(mat, axis=1)
        expressed_bins = np.sum(np.isfinite(mat) & (mat >= MIN_BIN_MEAN_EXPR), axis=1)
        valid_bins = np.sum(np.isfinite(mat), axis=1)
        original_pass = ((profile_mean >= MIN_PROFILE_MEAN_EXPR)
                         & (expressed_bins >= MIN_EXPRESSED_BINS)
                         & (valid_bins >= MIN_VALID_BINS_FOR_VARIATION))
        for i,gene in enumerate(genes):
            qc_records.append({"gene":gene,"condition":c,"sample":s,
                               "profile_mean_expr":profile_mean[i],
                               "expressed_bins":int(expressed_bins[i]),
                               "valid_bins":int(valid_bins[i]),
                               "sample_pass_original_QC":bool(original_pass[i]),
                               "sample_pass_applied":bool(original_pass[i]) if USE_GENE_QC
                                   else bool(valid_bins[i] >= MIN_VALID_BINS_FOR_VARIATION)})
    sample_qc = pd.DataFrame(qc_records)
    sample_qc.to_csv(output_file(prefix, "01_gene_sample_profile_QC.csv"),index=False)
    cq = (sample_qc.groupby(["gene","condition"], observed=True)
          .agg(n_samples=("sample","nunique"),
               n_valid_samples=("sample_pass_original_QC","sum"),
               valid_fraction=("sample_pass_original_QC","mean"),
               median_profile_mean_expr=("profile_mean_expr","median"),
               median_expressed_bins=("expressed_bins","median"))
          .reset_index())
    cq["condition_pass_original_QC"] = (cq["valid_fraction"] >= MIN_VALID_SAMPLE_FRACTION_PER_CONDITION)
    pass_table = cq.pivot(index="gene",columns="condition",values="condition_pass_original_QC").reindex(columns=CONDITIONS)
    original_keep = pass_table.fillna(False).all(axis=1)
    audit = pd.DataFrame({"gene":genes}).set_index("gene")
    for c in CONDITIONS:
        tmp = cq[cq["condition"].eq(c)].set_index("gene")
        audit[f"{c}_valid_fraction"] = tmp["valid_fraction"]
        audit[f"{c}_median_profile_mean_expr"] = tmp["median_profile_mean_expr"]
        audit[f"{c}_median_expressed_bins"] = tmp["median_expressed_bins"]
    audit["original_condition_QC_pass"] = original_keep.reindex(audit.index,fill_value=False)
    if MANUAL_GENES is None:
        audit["manual_pass"] = True
    else:
        requested = list(dict.fromkeys(str(g) for g in MANUAL_GENES))
        unknown = sorted(set(requested)-set(genes))
        if unknown:
            raise KeyError(f"MANUAL_GENES contains missing genes: {unknown}")
        audit["manual_pass"] = audit.index.isin(requested)
    audit["pass_all_filters"] = (audit["manual_pass"]
                                 & (audit["original_condition_QC_pass"] if USE_GENE_QC else True))
    audit = audit.reset_index()
    cq.to_csv(output_file(prefix, "02_gene_condition_QC.csv"),index=False)
    audit.to_csv(output_file(prefix, "03_gene_filter_QC_summary.csv"),index=False)
    return sample_qc,audit


def bh_adjust(pvalues):
    p = np.asarray(pvalues,float)
    q = np.full(len(p),np.nan,float)
    good = np.isfinite(p)
    if not good.any():
        return q
    pp = p[good]
    order = np.argsort(pp)
    ranks = np.arange(1,len(pp)+1)
    qq = np.minimum.accumulate((pp[order]*len(pp)/ranks)[::-1])[::-1]
    qq = np.clip(qq,0,1)
    mapped = np.empty(len(pp))
    mapped[order] = qq
    q[good] = mapped
    return q


def paired_stats(paired, target, axis_name, metric):
    rows=[]
    for a,b in [("WT","Heter"),("WT","Homo"),("Heter","Homo")]:
        x=paired[a].to_numpy(float); y=paired[b].to_numpy(float)
        if len(x) < 2 or np.allclose(y-x,0,atol=0,rtol=0):
            stat,p=np.nan,1.0 if len(x)>=2 else np.nan
        else:
            try:stat,p=wilcoxon(x,y)
            except ValueError:stat,p=np.nan,np.nan
        delta=y-x
        rows.append({"celltype":target,"axis":axis_name,"metric":metric,
                     "group1":a,"group2":b,"n_genes":len(delta),
                     "group1_median":np.nanmedian(x) if len(x) else np.nan,
                     "group2_median":np.nanmedian(y) if len(y) else np.nan,
                     "median_paired_delta_group2_minus_group1":np.nanmedian(delta) if len(delta) else np.nan,
                     "mean_paired_delta_group2_minus_group1":np.nanmean(delta) if len(delta) else np.nan,
                     "wilcoxon_statistic":stat,"p_value":p})
    df=pd.DataFrame(rows)
    df["q_BH_3comparisons"]=bh_adjust(df["p_value"])
    return df


In [ ]:
def fmt_significance(q):
    """BH-adjusted q -> *, **, ***, ****; non-significant -> ns."""
    if not np.isfinite(q):
        return None
    if q < 1e-4:
        return "****"
    if q < 1e-3:
        return "***"
    if q < 1e-2:
        return "**"
    if q < 0.05:
        return "*"
    return "ns" if SHOW_NS_SIGNIFICANCE else None


def new_ax(width=AXES_WIDTH_MM, height=AXES_HEIGHT_MM, top=TOP_MM):
    fw=LEFT_MM+width+RIGHT_MM; fh=BOTTOM_MM+height+top
    fig=plt.figure(figsize=(fw/25.4,fh/25.4))
    ax=fig.add_axes([LEFT_MM/fw,BOTTOM_MM/fh,width/fw,height/fh])
    return fig,ax


def finish(fig,path):
    try:
        fig.savefig(path,dpi=DPI,bbox_inches="tight",pad_inches=.02,transparent=False)
        if SHOW_FIGURES:plt.show()
    finally:
        plt.close(fig)


def label_name(metric):
    return {"relative_dynamic_range":"Relative spatial variation\n(max−min)/mean",
            "cv":"Spatial variation (CV)","dynamic_range":"Spatial dynamic range"}[metric]


def draw_sig(ax, stats, xloc=None, fixed_ylim=None):
    """Annotate significance using BH-adjusted q-values.

    With fixed_ylim, place brackets above the plotting area without changing
    data-axis limits. Otherwise, expand the upper limit for annotations.
    """
    pairs = [("WT", "Heter"), ("WT", "Homo"), ("Heter", "Homo")]
    pos = {c: i + 1 for i, c in enumerate(CONDITIONS)}
    if fixed_ylim is not None:
        ax.set_ylim(*fixed_ylim)
        trans = ax.get_xaxis_transform()
        for j, (a, b) in enumerate(pairs):
            row = stats[(stats.group1 == a) & (stats.group2 == b)]
            if row.empty:
                continue
            q = float(row.iloc[0]["q_BH_3comparisons"])
            star = fmt_significance(q)
            if star is None:
                continue
            y = 1.045 + 0.115 * j
            xa, xb = pos[a], pos[b]
            ax.plot([xa, xa, xb, xb], [y, y + .018, y + .018, y],
                    transform=trans, lw=.5, c='0.25', clip_on=False)
            ax.text((xa + xb) / 2, y + .025, star, transform=trans,
                    ha='center', va='bottom', fontsize=6, clip_on=False)
        ax.set_ylim(*fixed_ylim)
        return

    low, high = ax.get_ylim()
    span = max(high - low, 1e-8)
    height_base = high + .045 * span
    for j, (a, b) in enumerate(pairs):
        row = stats[(stats.group1 == a) & (stats.group2 == b)]
        if row.empty:
            continue
        q = float(row.iloc[0]["q_BH_3comparisons"])
        star = fmt_significance(q)
        if star is None:
            continue
        y = height_base + j * .13 * span
        xa, xb = pos[a], pos[b]
        ax.plot([xa, xa, xb, xb], [y, y + .014 * span, y + .014 * span, y],
                lw=.5, c='0.25', clip_on=False)
        ax.text((xa + xb) / 2, y + .020 * span, star,
                ha='center', va='bottom', fontsize=6, clip_on=False)
    ax.set_ylim(low, high + .53 * span)


def plot_gene_scatter(paired, stats, target, region, axis_name, metric, prefix):

    is_fixed_cv = (metric == "cv")
    fig, ax = new_ax(top=30 if is_fixed_cv else TOP_MM)
    positions = np.arange(1, len(CONDITIONS) + 1)
    box_data = [paired[c].to_numpy(float) for c in CONDITIONS]
    bp = ax.boxplot(box_data, positions=positions, widths=.55, patch_artist=True,
                    showfliers=False, medianprops=dict(color='black', linewidth=.7),
                    boxprops=dict(color='black', linewidth=.6), whiskerprops=dict(color='black', linewidth=.6),
                    capprops=dict(color='black', linewidth=.6))
    for patch, c in zip(bp['boxes'], CONDITIONS):
        patch.set_facecolor(CMAP_COLORS[c]); patch.set_alpha(.25)
    if SHOW_PAIRED_LINES:
        for row in paired[CONDITIONS].to_numpy(float):
            ax.plot(positions, row, c='0.82', linewidth=.25, alpha=.4, zorder=1)
    rng = np.random.default_rng(1)
    for xpos, c in zip(positions, CONDITIONS):
        vals = paired[c].to_numpy(float)
        ax.scatter(np.full(len(vals), xpos) + rng.normal(0, .07, len(vals)), vals,
                   s=1, c=CMAP_COLORS[c], alpha=.65, edgecolors='none', zorder=3)
    ax.set_xticks(positions, CONDITIONS)
    ax.set_ylabel(label_name(metric))
    title = f"{region} | {target} | axis {axis_name}\nn={len(paired)} paired genes"
    ax.set_title(title, y=1.44 if is_fixed_cv else None, pad=3)

    if is_fixed_cv:
        ymin, ymax = CV_SCATTER_YLIM
        ax.set_ylim(ymin, ymax)
        qc_rows = []
        for c in CONDITIONS:
            values = paired[c].to_numpy(float)
            finite = values[np.isfinite(values)]
            qc_rows.append({
                'region': region, 'celltype': target, 'axis': axis_name,
                'metric': metric, 'condition': c,
                'fixed_y_min': ymin, 'fixed_y_max': ymax,
                'n_finite_genes': len(finite),
                'n_below_y_min': int((finite < ymin).sum()),
                'n_above_y_max': int((finite > ymax).sum()),
                'max_original_cv': float(np.max(finite)) if len(finite) else np.nan,
            })
        pd.DataFrame(qc_rows).to_csv(
            output_file(prefix, '04_cv_fixed_y_range_QC.csv'), index=False)

        draw_sig(ax, stats, fixed_ylim=CV_SCATTER_YLIM)
    else:
        if PLOT_Y_MIN is not None or PLOT_Y_MAX is not None:
            yl = ax.get_ylim()
            ax.set_ylim(PLOT_Y_MIN if PLOT_Y_MIN is not None else yl[0],
                        PLOT_Y_MAX if PLOT_Y_MAX is not None else yl[1])
        draw_sig(ax, stats)

    ax.tick_params(width=.5, length=2, pad=1)
    for sp in ax.spines.values(): sp.set_linewidth(.5)
    finish(fig, output_file(prefix, f"04_gene_scatter_boxplot_{metric}.pdf"))


def plot_delta(paired,stats,region,target,axis_name,metric,prefix):
    fig,ax=new_ax(width=52,top=12)
    pairs=[("Heter","WT"),("Homo","WT")]
    rng=np.random.default_rng(42)
    deltas=[]
    for i,(a,b) in enumerate(pairs):
        d=(paired[a]-paired[b]).to_numpy(float)
        deltas.append(d)
        xx=np.full(len(d),i+1)+rng.normal(0,.065,len(d))
        ax.scatter(xx,d,s=6,c=CMAP_COLORS[a],alpha=.42,edgecolors='none',rasterized=False)
        ax.scatter([i+1],[np.median(d)],s=20,c='black',marker='_',zorder=7)
        npos=int(np.sum(d>0));nneg=int(np.sum(d<0));n0=int(np.sum(d==0))
        ax.text(i+1,.98,f"median Δ={np.median(d):+.3g}\n↑ {npos} / ↓ {nneg} / = {n0}",
                ha='center',va='top',fontsize=6,transform=ax.get_xaxis_transform())
    ax.axhline(0,c='0.3',lw=.6)
    ax.set_xticks([1,2],["Heter − WT","Homo − WT"])
    ax.set_ylabel("Δ " + label_name(metric).split('\n')[0])
    ax.set_title(f"{region} | {target} | axis {axis_name}\nGene-wise change vs WT",pad=3)
    ax.margins(y=.15);ax.tick_params(width=.5,length=2,pad=1)
    for sp in ax.spines.values():sp.set_linewidth(.5)
    finish(fig,output_file(prefix, f"05_gene_direction_vs_WT_{metric}.pdf"))


def animal_summary(sample_var, paired_genes, metric,region,target,axis_name,prefix):

    dat=sample_var[sample_var['gene'].isin(paired_genes)&sample_var['metric'].eq(metric)]
    rec=[]
    for (c,s),g in dat.groupby(['condition','sample'],observed=True):
        v=g['variation'].dropna().to_numpy(float)
        rec.append({'condition':c,'sample':s,'n_valid_genes':len(v),
                    'median_variation':float(np.median(v)) if len(v) else np.nan,
                    'mean_variation':float(np.mean(v)) if len(v) else np.nan})
    df=pd.DataFrame(rec)
    df.to_csv(output_file(prefix, f"06_animal_summary_{metric}.csv"),index=False)
    tests=[]
    for a,b in [('WT','Heter'),('WT','Homo'),('Heter','Homo')]:
        va=df.loc[df.condition.eq(a),'median_variation'].dropna().to_numpy(float)
        vb=df.loc[df.condition.eq(b),'median_variation'].dropna().to_numpy(float)
        if len(va)>=2 and len(vb)>=2:
            u,p=mannwhitneyu(va,vb,alternative='two-sided',method='auto')
        else:u,p=np.nan,np.nan
        tests.append({'group1':a,'group2':b,'n_group1':len(va),'n_group2':len(vb),
                      'mannwhitney_u':u,'p_value':p})
    tests=pd.DataFrame(tests)
    tests['q_BH_3comparisons']=bh_adjust(tests.p_value)
    tests.to_csv(output_file(prefix, f"07_animal_mannwhitney_{metric}.csv"),index=False)
    if not DRAW_ANIMAL_LEVEL:return
    fig,ax=new_ax(top=TOP_MM)
    rng=np.random.default_rng(35)
    for j,c in enumerate(CONDITIONS,1):
        vals=df.loc[df.condition.eq(c),'median_variation'].dropna().to_numpy(float)
        if not len(vals):continue
        ax.scatter(np.full(len(vals),j)+rng.normal(0,.055,len(vals)), vals,
                   c=CMAP_COLORS[c],s=19,alpha=.8,zorder=3)
        ax.errorbar(j,vals.mean(),yerr=vals.std(ddof=1)/np.sqrt(len(vals)) if len(vals)>1 else 0,
                    color='black',linewidth=.7,fmt='_',capsize=2,zorder=4)
    ax.set_xticks([1,2,3],CONDITIONS)
    ax.set_ylabel('Sample median '+label_name(metric).split('\n')[0])
    ax.set_title(f"{region} | {target} | axis {axis_name}\nOne dot = one animal",pad=3)
    draw_sig(ax,tests)
    ax.tick_params(width=.5,length=2,pad=1)
    for sp in ax.spines.values():sp.set_linewidth(.5)
    finish(fig,output_file(prefix, f"08_animal_level_{metric}.pdf"))


In [ ]:
def select_celltype(obs, target):
    ct=obs[CELLTYPE_COL].astype(str)
    requested={'D1':'MSN_D1','D2':'MSN_D2','MSN_D1':'MSN_D1','MSN_D2':'MSN_D2'}[target.upper()]
    if requested in set(ct):
        return ct.eq(requested).to_numpy()
    pattern=rf'(?:^|_){re.escape(target.upper())}(?:$|_)'
    return ct.str.upper().str.contains(pattern,regex=True).to_numpy()


def process_one(backed,region,target,axis_name):
    celltype={'D1':'MSN_D1','D2':'MSN_D2'}[target.upper()]
    prefix=f'{region}_{target.upper()}_{axis_name}'
    obs=backed.obs
    sel=(obs[REGION_COL].astype(str).eq(region).to_numpy()
         & select_celltype(obs,target)
         & obs[COND_COL].astype(str).isin(CONDITIONS).to_numpy())
    idx=np.where(sel)[0]
    if not len(idx):
        raise ValueError(f'{region}/{target}/{axis_name}: no cells passed selection; check region_str_manual/refined_major_cluster')

    ad=backed[idx,:].to_memory()
    if EXPR_LAYER not in ad.layers:
        raise KeyError(f'{region}/{target}/{axis_name}: missing layer[{EXPR_LAYER!r}]')
    genes=ad.var_names.astype(str).to_numpy()
    if len(set(genes))!=len(genes):raise ValueError('Input gene names contain duplicates')
    print(f'\n{region}/{target}/{axis_name}: {len(idx)} cells; {len(genes)} genes')
    sample_profiles,bin_qc=make_sample_profiles(ad,axis_name)
    bin_qc.to_csv(output_file(prefix, '00_sample_bin_counts.csv'),index=False)
    sample_qc,gene_qc=gene_profile_qc(sample_profiles,genes,prefix)
    kept_genes=gene_qc.loc[gene_qc.pass_all_filters,'gene'].tolist()
    gene_idx={g:i for i,g in enumerate(genes)}
    print(f'  QC pass: {len(kept_genes)}/{len(genes)} genes (enabled={USE_GENE_QC})')
    if not kept_genes:
        raise ValueError(f'{region}/{target}/{axis_name}: no genes remain after filtering; check thresholds and USE_GENE_QC')
    qc_ok=sample_qc.set_index(['gene','condition','sample'])['sample_pass_applied'].to_dict()
    sample_rows=[]
    for (c,s),mat in sample_profiles.items():
        for gene in kept_genes:
            ii=gene_idx[gene]
            for metric in METRICS:
                passed=bool(qc_ok.get((gene,c,s),False))
                value=calc_variation(mat[ii],metric) if passed else np.nan
                sample_rows.append({'gene':gene,'condition':c,'sample':s,'metric':metric,
                                    'variation':value,'sample_pass':passed})
    sample_var=pd.DataFrame(sample_rows)
    sample_var.to_csv(output_file(prefix, '09_gene_sample_all_metric_variation.csv'),index=False)
    group_ct={c:sum(cc==c for cc,s in sample_profiles) for c in CONDITIONS}
    minimum_n={c:(max(1,math.ceil(group_ct[c]*MIN_VALID_SAMPLE_FRACTION_PER_CONDITION))
                  if USE_GENE_QC else 1) for c in CONDITIONS}
    results=[]
    for metric in METRICS:
        metric_prefix=f'{prefix}_{metric}'
        temp=sample_var[sample_var.metric.eq(metric)].copy()
        gvar=(temp.groupby(['gene','condition'],observed=True)['variation']
              .agg(['mean','std','count','median']).reset_index())
        gvar['sem']=gvar['std']/np.sqrt(gvar['count'])
        gvar=gvar.rename(columns={'mean':'variation'})
        gvar['enough_samples']=[int(r['count'])>=minimum_n[r['condition']] for _,r in gvar.iterrows()]
        gvar.loc[~gvar.enough_samples,'variation']=np.nan
        gvar.to_csv(output_file(metric_prefix, '10_gene_condition_one_point_per_gene.csv'),index=False)
        wide=(gvar.pivot(index='gene',columns='condition',values='variation')
              .reindex(columns=CONDITIONS))
        paired=wide.dropna(subset=CONDITIONS).copy()
        paired.to_csv(output_file(metric_prefix, '11_gene_WT_Heter_Homo_paired.csv'),index_label='gene')
        print(f'  {metric}: {len(paired)} completely paired genes')
        if len(paired)<2:
            print('  WARNING: fewer than 2 fully paired genes; no test or plot will be generated.')
            results.append({'region':region,'celltype':celltype,'axis':axis_name,'metric':metric,
                            'n_cells':len(idx),'n_genes_before':len(genes),
                            'n_after_QC':len(kept_genes),'n_paired':len(paired),'status':'few genes'})
            continue
        delta=paired.copy()
        delta['Heter_minus_WT']=delta.Heter-delta.WT
        delta['Homo_minus_WT']=delta.Homo-delta.WT
        delta['Homo_minus_Heter']=delta.Homo-delta.Heter
        delta.to_csv(output_file(metric_prefix, '12_gene_delta_signed_vs_WT.csv'),index_label='gene')
        stats=paired_stats(paired,target,axis_name,metric)
        stats.to_csv(output_file(metric_prefix, '13_paired_gene_wilcoxon_BH.csv'),index=False)
        print(stats[['group1','group2','n_genes','p_value','q_BH_3comparisons']].to_string(index=False))
        plot_gene_scatter(paired,stats,target,region,axis_name,metric,metric_prefix)
        plot_delta(paired,stats,region,target,axis_name,metric,metric_prefix)
        animal_summary(temp,paired.index.tolist(),metric,region,target,axis_name,metric_prefix)
        results.append({'region':region,'celltype':celltype,'axis':axis_name,'metric':metric,
                        'n_cells':len(idx),'n_genes_before':len(genes),'n_after_QC':len(kept_genes),
                        'n_paired':len(paired),'status':'saved'})
        gc.collect()

    pd.DataFrame([{'source_file':str(SOURCE_FILE),'region':region,'celltype':celltype,
                   'axis':axis_name,'expression_layer':EXPR_LAYER,'n_bins':N_BINS,
                   'min_valid_bins':MIN_VALID_BINS_FOR_VARIATION,
                   'use_gene_qc':USE_GENE_QC,'manual_genes':str(MANUAL_GENES),
                   'min_bin_mean_expr':MIN_BIN_MEAN_EXPR,'min_expressed_bins':MIN_EXPRESSED_BINS,
                   'min_profile_mean_expr':MIN_PROFILE_MEAN_EXPR,
                   'min_valid_sample_fraction':MIN_VALID_SAMPLE_FRACTION_PER_CONDITION,
                   'n_cells':len(idx),'n_genes':len(genes),'n_gene_pass':len(kept_genes)}]
                 ).to_csv(output_file(prefix, '_analysis_parameters.csv'),index=False)
    del ad,sample_profiles,sample_qc,sample_var
    gc.collect()
    return results


In [ ]:
backed=sc.read_h5ad(SOURCE_FILE,backed='r')
try:
    cols=[REGION_COL,CELLTYPE_COL,COND_COL,SAMPLE_COL,*[AXIS_COLUMNS[a] for a in TARGET_AXES]]
    missing=[c for c in cols if c not in backed.obs.columns]
    if missing:raise KeyError(f'obs is missing required columns: {missing}')
    if EXPR_LAYER not in backed.layers:
        raise KeyError(f'Source data is missing layer[{EXPR_LAYER!r}]; scaled adata.X cannot be used. Available layers: {list(backed.layers.keys())}')
    result=[]
    for region in TARGET_REGIONS:
        for target in TARGET_CELLTYPES:
            for axis in TARGET_AXES:
                result.extend(process_one(backed,region,target,axis))
    table=pd.DataFrame(result)
    table.to_csv(OUTPUT_ROOT/'_all_analysis_summary.csv',index=False)
    print('\nALL DONE:',OUTPUT_ROOT)
    display(table)
finally:
    if getattr(backed,'file',None) is not None:
        backed.file.close()
    gc.collect()


## Panel E | Condition-colored five-axis CV radar

Run the panel D CV-distribution workflow first. The radar rebuilds animal summaries from its flat CSVs when the preferred summary tables configured in the notebook are absent. Existing preferred tables take precedence. The panel E overlay is `Figure4_CV_five_axes_D1_D2_radar_overlay_6_curves.pdf`; output filenames use earlier figure numbering. The helper cell also exports a three-panel radar.


In [ ]:
# Imports and settings.
from pathlib import Path
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


BASE = Path(r"F:\BASSFISH_HD\Figure\Figure_4\Panel\1000gene")
PREFERRED_ANIMAL_TABLES = [
    BASE / "D1_D2_CV_RDR_5axes_paired_comparison_smallN_exact" / "04_animal_paired_CV_RDR_summary.csv",
    BASE / "D1_D2_CV_RDR_5axes_paired_comparison" / "04_animal_paired_CV_RDR_summary.csv",
]

FALLBACK_SOURCE_FLAT = BASE / "gene_axis_overall_variation_original_QC_optional_RDR_CV_FLAT"
FALLBACK_SOURCE_HIERARCHICAL = BASE / "gene_axis_overall_variation_original_QC_optional_RDR_CV"
ALLOW_REBUILD_FROM_SOURCE_CSV = True

MATCH_BOTH_METRICS_WHEN_REBUILDING = True
ANIMAL_GENE_AGG = "median"
MIN_PAIRED_GENES_PER_ANIMAL = 1

OUT_DIR = BASE / "D1_D2_CV_five_axes_radar_and_scatter"


AXIS_ORDER = ["d-x", "d-y", "n-x", "n-y", "o-x"]
COND_ORDER = ["WT", "Heter", "Homo"]
D1_COLOR, D2_COLOR = "#4477AA", "#EE9955"
CONDITION_COLORS = {"WT": "#4cd964", "Heter": "#ff9500", "Homo": "#ff3b30"}
COND_MARKERS = {"WT": "o", "Heter": "o", "Homo": "o"}
AXIS_MARKERS = {"d-x": "o", "d-y": "s", "n-x": "D", "n-y": "^", "o-x": "v"}
RADAR_AGG = "mean"
RADAR_RMAX = None
RADAR_RGRID_COUNT = 4
RADAR_SHOW_ZERO_ONE = True
RADAR_ONE_LABEL_ANGLE_DEG = 90
REQUIRE_COMPLETE_RADAR = True


RADAR_PANEL_WIDTH_MM = 58.0
RADAR_PANEL_HEIGHT_MM = 58.0
RADAR_PANEL_GAP_MM = 11.0
RADAR_MARGIN_LEFT_MM = 8.0
RADAR_MARGIN_RIGHT_MM = 8.0
RADAR_MARGIN_TOP_MM = 20.0
RADAR_MARGIN_BOTTOM_MM = 12.0
RADAR_LINE_WIDTH_PT = 1.2
RADAR_MARKER_SIZE_PT = 2.0
RADAR_FILL_ALPHA = 0.10

RADAR6_AXES_WIDTH_MM = 73.0
RADAR6_AXES_HEIGHT_MM = 73.0
RADAR6_MARGIN_LEFT_MM = 11.0
RADAR6_MARGIN_RIGHT_MM = 40.0
RADAR6_MARGIN_TOP_MM = 13.0
RADAR6_MARGIN_BOTTOM_MM = 10.0
RADAR6_LINE_WIDTH_PT = 1.3
RADAR6_MARKER_SIZE_PT = 2.4
RADAR6_FILL_ALPHA = 0.0
RADAR6_SHOW_LEGENDS = True

CELLTYPE_LINE_STYLES = {"D1": "-", "D2": (0, (3, 2))}
CONDITION_RADAR_MARKERS = {"WT": "o", "Heter": "s", "Homo": "^"}

RADAR_GRID_COLOR = "#D1D5DB"
RADAR_SHOW_RADIAL_TICKS = True


SCATTER_AXES_WIDTH_MM = 71.0
SCATTER_AXES_HEIGHT_MM = 67.0
SCATTER_MARGIN_LEFT_MM = 17.0
SCATTER_MARGIN_RIGHT_MM = 23.0
SCATTER_MARGIN_TOP_MM = 12.0
SCATTER_MARGIN_BOTTOM_MM = 17.0
SCATTER_POINT_SIZE_PT2 = 19.0
SCATTER_POINT_ALPHA = 0.80
SCATTER_EDGE_WIDTH_PT = 0.35
SCATTER_XY_MIN = 0.0
SCATTER_XY_MAX = None
SCATTER_REF_LINE_COLOR = "#CB332D"
SCATTER_REF_LINE_STYLE = "--"
SCATTER_REF_LINE_WIDTH_PT = 0.9
SCATTER_SHOW_CONDITION_LEGEND = True
SCATTER_SHOW_AXIS_LEGEND = True
LABEL_EACH_POINT = False


SCATTER15_AXES_WIDTH_MM = 71.0
SCATTER15_AXES_HEIGHT_MM = 67.0
SCATTER15_MARGIN_LEFT_MM = 17.0
SCATTER15_MARGIN_RIGHT_MM = 23.0
SCATTER15_MARGIN_TOP_MM = 12.0
SCATTER15_MARGIN_BOTTOM_MM = 17.0
SCATTER15_POINT_SIZE_PT2 = 28.0
SCATTER15_POINT_ALPHA = 0.92
SCATTER15_EDGE_WIDTH_PT = 0.35
SCATTER15_XY_MIN = SCATTER_XY_MIN
SCATTER15_XY_MAX = SCATTER_XY_MAX
SCATTER15_SHOW_CONDITION_LEGEND = True
SCATTER15_SHOW_AXIS_LEGEND = True
SCATTER15_LABEL_POINTS = False


FONT = "Arial"
FONT_SIZE_PT = 7
DPI = 600
SHOW_FIGURES = False
PDF_TIGHT_BBOX = False
plt.rcParams.update({
    "font.family": FONT, "font.size": FONT_SIZE_PT,
    "axes.titlesize": FONT_SIZE_PT, "axes.labelsize": FONT_SIZE_PT,
    "xtick.labelsize": FONT_SIZE_PT, "ytick.labelsize": FONT_SIZE_PT,
    "legend.fontsize": FONT_SIZE_PT, "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.linewidth": 0.6,
})
assert RADAR_AGG in {"mean", "median"}
assert ANIMAL_GENE_AGG in {"mean", "median"}
assert len(AXIS_ORDER) == 5, "This plot requires five spatial axes."
print("Preferred previous animal summary:", *[str(p) for p in PREFERRED_ANIMAL_TABLES], sep="\n  ")
print("Output:", OUT_DIR)


In [ ]:
def source_file_triplet(axis_id, ct):
    region, axis = axis_id.split("-")
    prefix = f"{region}_{ct}_{axis}"
    flat = {
        "sample": FALLBACK_SOURCE_FLAT / f"{prefix}_09_gene_sample_all_metric_variation.csv",
        "cv": FALLBACK_SOURCE_FLAT / f"{prefix}_cv_11_gene_WT_Heter_Homo_paired.csv",
        "rdr": FALLBACK_SOURCE_FLAT / f"{prefix}_relative_dynamic_range_11_gene_WT_Heter_Homo_paired.csv",
    }
    old = FALLBACK_SOURCE_HIERARCHICAL / axis / f"region_{region}" / f"celltype_MSN_{ct}"
    hierarchical = {
        "sample": old / "09_gene_sample_all_metric_variation.csv",
        "cv": old / "cv" / "11_gene_WT_Heter_Homo_paired.csv",
        "rdr": old / "relative_dynamic_range" / "11_gene_WT_Heter_Homo_paired.csv",
    }
    need = ("sample", "cv", "rdr") if MATCH_BOTH_METRICS_WHEN_REBUILDING else ("sample", "cv")
    for group in (flat, hierarchical):
        if all(group[k].is_file() for k in need):
            return {k: group[k] for k in need}
    raise FileNotFoundError(
        f"{axis_id} {ct} is missing CV/RDR inputs. Run the CV-distribution workflow first.\n"
        + "\n".join(str(flat[k]) for k in need)
    )


def _read_gene_names(path):
    genes = pd.read_csv(path, usecols=["gene"], dtype={"gene": "string"})["gene"].dropna()
    if genes.duplicated().any():
        raise ValueError(f"Duplicate genes: {path}")
    return set(genes.astype(str))


def rebuild_animal_cv():
    """Rebuild animal CV summaries from matched D1/D2 genes."""
    frames, gene_audits, manifest = [], [], []
    for axis_id in AXIS_ORDER:
        gene_sets, csv_inputs = [], {}
        for ct in ["D1", "D2"]:
            files = source_file_triplet(axis_id, ct)
            csv_inputs[ct] = files
            for role, path in files.items():
                manifest.append({"axis_id": axis_id, "celltype": ct, "role": role, "input_path": str(path)})
                if role in ("cv", "rdr"):
                    gene_sets.append(_read_gene_names(path))
        common = set.intersection(*gene_sets)
        if not common:
            raise ValueError(f"{axis_id}: no shared D1/D2 genes passed QC.")
        gene_audits.append({"axis_id": axis_id, "n_common_qc_genes": len(common)})
        for ct in ("D1", "D2"):
            path = csv_inputs[ct]["sample"]
            needed = ["gene", "condition", "sample", "metric", "variation", "sample_pass"]
            ss = pd.read_csv(path, usecols=needed,
                             dtype={"gene": "string", "condition": "string", "sample": "string", "metric": "string"})
            if ss.duplicated(["gene", "condition", "sample", "metric"]).any():
                raise ValueError(f"Duplicate gene/condition/sample/metric entries: {path}")
            ss = ss.loc[ss["gene"].isin(common) & ss["condition"].isin(COND_ORDER) & ss["metric"].eq("cv")].copy()
            ss["variation"] = pd.to_numeric(ss["variation"], errors="coerce")
            passed = ss["sample_pass"].astype(str).str.lower().isin(("true", "1", "yes"))
            ss.loc[~passed, "variation"] = np.nan
            ss = ss[["gene", "condition", "sample", "variation"]].copy()
            ss["axis_id"] = axis_id
            ss["celltype"] = ct
            frames.append(ss)
    raw = pd.concat(frames, ignore_index=True)
    keys = ["axis_id", "condition", "sample", "gene"]
    d1 = raw[raw.celltype.eq("D1")][keys + ["variation"]].rename(columns={"variation": "D1_gene_value"})
    d2 = raw[raw.celltype.eq("D2")][keys + ["variation"]].rename(columns={"variation": "D2_gene_value"})
    paired = d1.merge(d2, on=keys, how="inner", validate="one_to_one")
    paired = paired.replace([np.inf, -np.inf], np.nan).dropna(subset=["D1_gene_value", "D2_gene_value"])
    if paired.empty:
        raise ValueError("No valid D1/D2 CV pairs matched by animal, spatial axis, and gene.")
    agg_func = "median" if ANIMAL_GENE_AGG == "median" else "mean"
    animal = (paired.groupby(["axis_id", "condition", "sample"], as_index=False)
                   .agg(n_paired_genes=("gene", "nunique"),
                        D1=("D1_gene_value", agg_func), D2=("D2_gene_value", agg_func)))
    animal["passed_min_genes"] = animal["n_paired_genes"] >= MIN_PAIRED_GENES_PER_ANIMAL
    animal.loc[~animal.passed_min_genes, ["D1", "D2"]] = np.nan
    animal["metric"] = "cv"
    animal["delta_D2_minus_D1"] = animal["D2"] - animal["D1"]
    return animal, pd.DataFrame(manifest), pd.DataFrame(gene_audits)


def read_previous_animal_summary(path):
    raw = pd.read_csv(path, dtype={"condition": "string", "sample": "string", "metric": "string"})
    required = {"condition", "sample", "metric", "D1", "D2"}
    if not required.issubset(raw.columns):
        raise ValueError(f"Existing table {path} is missing columns {required - set(raw.columns)}")
    if "axis_id" not in raw.columns:
        if {"region", "axis"}.issubset(raw.columns):
            raw["axis_id"] = raw["region"].astype(str) + "-" + raw["axis"].astype(str)
        else:
            raise ValueError(f"Existing table {path} lacks axis_id or region/axis")
    raw["metric"] = raw["metric"].astype(str).str.lower()
    df = raw.loc[raw["metric"].eq("cv")].copy()
    if "passed_min_genes" in df.columns:
        good = df["passed_min_genes"].astype(str).str.lower().isin(("true", "1", "yes"))
        df = df.loc[good].copy()
    return df


def load_animal_cv():
    for p in PREFERRED_ANIMAL_TABLES:
        if p.is_file():
            d = read_previous_animal_summary(p)
            print("Loaded existing matched animal CV:", p)
            return d, pd.DataFrame([{"axis_id": "ALL", "celltype": "D1+D2", "role": "paired_animal_summary", "input_path": str(p)}]), pd.DataFrame(), "precomputed_animal_summary"
    if not ALLOW_REBUILD_FROM_SOURCE_CSV:
        raise FileNotFoundError("Completed 04_animal_paired_CV_RDR_summary.csv not found.")
    print("Precomputed animal summary not found; rebuilding from original CSV (not h5ad).")
    d, manifest, qc = rebuild_animal_cv()
    return d, manifest, qc, "recomputed_from_original_variation_csv"


In [ ]:
animal, input_manifest, shared_gene_qc, data_origin = load_animal_cv()
needed = ["axis_id", "condition", "sample", "D1", "D2"]
missing = set(needed) - set(animal.columns)
if missing:
    raise ValueError(f"Input is missing required columns: {missing}")
for col in ["D1", "D2"]:
    animal[col] = pd.to_numeric(animal[col], errors="coerce")
animal["condition"] = animal["condition"].astype(str).str.strip()
animal["sample"] = animal["sample"].astype(str).str.strip()
animal["axis_id"] = animal["axis_id"].astype(str).str.strip()
pairs = animal.loc[
    animal["condition"].isin(COND_ORDER) & animal["axis_id"].isin(AXIS_ORDER)
].copy()
if pairs.duplicated(["condition", "sample", "axis_id"]).any():
    bad = pairs.loc[pairs.duplicated(["condition", "sample", "axis_id"], keep=False), ["condition", "sample", "axis_id"]]
    raise ValueError(f"Duplicate results for the same animal/condition/axis:\n{bad.to_string(index=False)}")
pairs = pairs.replace([np.inf, -np.inf], np.nan).dropna(subset=["D1", "D2"]).copy()
if pairs.empty:
    raise ValueError("No valid D1/D2 pairs for condition + sample + axis.")
if (pairs[["D1", "D2"]].to_numpy(float) < 0).any():
    raise ValueError("Input contains CV < 0; check the data first.")
pairs["delta_D2_minus_D1"] = pairs["D2"] - pairs["D1"]
pairs["D2_below_D1"] = pairs["D2"] < pairs["D1"]
pairs["D2_equal_D1"] = np.isclose(pairs["D2"], pairs["D1"], rtol=0, atol=1e-12)
pairs["D2_above_D1"] = pairs["D2"] > pairs["D1"]
pairs["axis_id"] = pd.Categorical(pairs["axis_id"], categories=AXIS_ORDER, ordered=True)
pairs["condition"] = pd.Categorical(pairs["condition"], categories=COND_ORDER, ordered=True)
pairs = pairs.sort_values(["condition", "axis_id", "sample"]).reset_index(drop=True)

radar_summary = (pairs.groupby(["condition", "axis_id"], observed=True)[["D1", "D2"]]
                      .agg(RADAR_AGG).reset_index())
pair_counts = (pairs.groupby(["condition", "axis_id"], observed=True)
                    .agg(n_matched_animal_axis=("sample", "nunique"),
                         n_gene_min=("n_paired_genes", "min") if "n_paired_genes" in pairs.columns else ("sample", "size"))
                    .reset_index())
radar_summary = radar_summary.merge(pair_counts, on=["condition", "axis_id"], how="left", validate="one_to_one")

missing_axes = [(condition, ax) for condition in COND_ORDER for ax in AXIS_ORDER
                if not ((radar_summary.condition.astype(str) == condition) & (radar_summary.axis_id.astype(str) == ax)).any()]
if missing_axes and REQUIRE_COMPLETE_RADAR:
    raise ValueError("Radar plot lacks paired data for condition-by-axis groups: " + ", ".join(f"{c}/{a}" for c, a in missing_axes))

OUT_DIR.mkdir(parents=True, exist_ok=True)
pairs.to_csv(OUT_DIR / "00_D1_D2_CV_paired_animal_axis_values.csv", index=False, encoding="utf-8-sig")
radar_summary.to_csv(OUT_DIR / "01_radar_condition_axis_mean_CV.csv", index=False, encoding="utf-8-sig")
input_manifest.to_csv(OUT_DIR / "02_input_file_audit.csv", index=False, encoding="utf-8-sig")
if not shared_gene_qc.empty:
    shared_gene_qc.to_csv(OUT_DIR / "03_rebuilt_common_gene_QC.csv", index=False, encoding="utf-8-sig")
print(f"Data source: {data_origin}; paired axis-animal points: {len(pairs)}")
print(radar_summary.to_string(index=False))


In [ ]:
def _save_figure(fig, path):
    kwargs = {"dpi": DPI, "facecolor": "white"}
    if PDF_TIGHT_BBOX:
        kwargs.update({"bbox_inches": "tight", "pad_inches": 0.02})
    fig.savefig(path, **kwargs)
    if SHOW_FIGURES:
        plt.show()
    plt.close(fig)


def make_radar_pdf(data):
    w = RADAR_MARGIN_LEFT_MM + 3*RADAR_PANEL_WIDTH_MM + 2*RADAR_PANEL_GAP_MM + RADAR_MARGIN_RIGHT_MM
    h = RADAR_MARGIN_BOTTOM_MM + RADAR_PANEL_HEIGHT_MM + RADAR_MARGIN_TOP_MM
    if abs(RADAR_PANEL_WIDTH_MM - RADAR_PANEL_HEIGHT_MM) > 0.1:
        print("Note: radar panel is not square; equal aspect scaling preserves the pentagon shape.")
    fig = plt.figure(figsize=(w/25.4, h/25.4))
    all_values = data[["D1", "D2"]].to_numpy(float)
    real_max = float(np.nanmax(all_values))
    if RADAR_RMAX is None:
        radar_max = max(0.1, math.ceil(real_max*1.08*10)/10)
    else:
        radar_max = float(RADAR_RMAX)
    if radar_max <= 0:
        raise ValueError("RADAR_RMAX must be greater than zero")
    if np.any(all_values > radar_max):
        raise ValueError(f"CV exceeds RADAR_RMAX={radar_max}; increase the upper limit to display the full radar range.")
    theta = np.pi/2 - np.arange(len(AXIS_ORDER))*2*np.pi/len(AXIS_ORDER)
    vectors = np.column_stack([np.cos(theta), np.sin(theta)])
    for ic, cond in enumerate(COND_ORDER):
        x0 = RADAR_MARGIN_LEFT_MM + ic*(RADAR_PANEL_WIDTH_MM + RADAR_PANEL_GAP_MM)
        ax = fig.add_axes([x0/w, RADAR_MARGIN_BOTTOM_MM/h,
                           RADAR_PANEL_WIDTH_MM/w, RADAR_PANEL_HEIGHT_MM/h])
        ax.set_aspect('equal')
        ax.set_xlim(-1.23, 1.23);ax.set_ylim(-1.23, 1.23); ax.axis('off')
        for k in range(1, RADAR_RGRID_COUNT+1):
            r = k/RADAR_RGRID_COUNT
            v = np.vstack([vectors*r, vectors[:1]*r])
            ax.plot(v[:,0], v[:,1], color=RADAR_GRID_COLOR, lw=0.55, zorder=0)
            if RADAR_SHOW_RADIAL_TICKS and k in (2, RADAR_RGRID_COUNT):
                ax.text(0.02, r + 0.01, f"{radar_max*r:.2g}", ha='left', va='bottom',
                        fontsize=FONT_SIZE_PT-1.5, color='#71717A')
        for v in vectors:
            ax.plot([0,v[0]], [0,v[1]], color=RADAR_GRID_COLOR, lw=0.45, zorder=0)
        for i, name in enumerate(AXIS_ORDER):
            v = vectors[i]*1.13
            ax.text(v[0], v[1], name, fontsize=FONT_SIZE_PT, ha='center', va='center')
        sub = data[data.condition.astype(str).eq(cond)].set_index('axis_id').reindex(AXIS_ORDER)
        for ct, color in [('D1', D1_COLOR), ('D2', D2_COLOR)]:
            arr = sub[ct].to_numpy(float)
            if not np.isfinite(arr).all():
                if REQUIRE_COMPLETE_RADAR:
                    raise ValueError(f"{cond} {ct}: radar plot has missing axis data")
                print(f"Skipping incomplete trace: {cond}/{ct}")
                continue
            xy = vectors*(arr/radar_max)[:,None]
            closed = np.vstack([xy, xy[:1]])
            ax.fill(closed[:,0], closed[:,1], color=color, alpha=RADAR_FILL_ALPHA, zorder=2)
            ax.plot(closed[:,0], closed[:,1], lw=RADAR_LINE_WIDTH_PT, color=color,
                    marker='o', markersize=RADAR_MARKER_SIZE_PT, zorder=3)
        ax.set_title(cond, fontsize=FONT_SIZE_PT+1, pad=10)
    fig.legend(handles=[Line2D([0],[0],color=D1_COLOR,lw=1.6,label='MSN-D1'),
                        Line2D([0],[0],color=D2_COLOR,lw=1.6,label='MSN-D2')],
               loc='upper center', ncol=2, frameon=False,
               bbox_to_anchor=(.5,1-6.0/h))
    out = OUT_DIR/"Figure4_CV_five_axes_D1_D2_radar_WT_Heter_Homo.pdf"
    _save_figure(fig,out)
    print("Radar shared Rmax =",radar_max, "Saved:",out)
    return out,radar_max

radar_pdf, radar_max_used = make_radar_pdf(radar_summary)


In [ ]:
def make_six_curves_radar_pdf(data, rmax):
    w = RADAR6_MARGIN_LEFT_MM + RADAR6_AXES_WIDTH_MM + RADAR6_MARGIN_RIGHT_MM
    h = RADAR6_MARGIN_BOTTOM_MM + RADAR6_AXES_HEIGHT_MM + RADAR6_MARGIN_TOP_MM
    fig = plt.figure(figsize=(w / 25.4, h / 25.4))
    ax = fig.add_axes([
        RADAR6_MARGIN_LEFT_MM / w,
        RADAR6_MARGIN_BOTTOM_MM / h,
        RADAR6_AXES_WIDTH_MM / w,
        RADAR6_AXES_HEIGHT_MM / h,
    ])
    ax.set_aspect('equal')
    ax.set_xlim(-1.23, 1.23)
    ax.set_ylim(-1.23, 1.23)
    ax.axis('off')

    theta = np.pi / 2 - np.arange(len(AXIS_ORDER)) * 2 * np.pi / len(AXIS_ORDER)
    vectors = np.column_stack([np.cos(theta), np.sin(theta)])
    for k in range(1, RADAR_RGRID_COUNT + 1):
        r = k / RADAR_RGRID_COUNT
        xy = np.vstack([vectors * r, vectors[:1] * r])
        ax.plot(xy[:, 0], xy[:, 1], color=RADAR_GRID_COLOR, lw=0.55, zorder=0)
        if RADAR_SHOW_RADIAL_TICKS and k in (2, RADAR_RGRID_COUNT):
            ax.text(0.02, r + 0.01, f'{rmax * r:.2g}', ha='left', va='bottom',
                    fontsize=FONT_SIZE_PT - 1.5, color='#71717A')
    for v in vectors:
        ax.plot([0, v[0]], [0, v[1]], color=RADAR_GRID_COLOR, lw=0.45, zorder=0)
    for axis_idx, name in enumerate(AXIS_ORDER):
        text_xy = vectors[axis_idx] * 1.13
        ax.text(text_xy[0], text_xy[1], name, fontsize=FONT_SIZE_PT,
                ha='center', va='center')

    all_values = data[['D1','D2']].to_numpy(dtype=float)
    if np.nanmax(all_values) > rmax + 1e-10:
        raise ValueError(f'Six-curve radar: CV exceeds the shared upper limit {rmax}')

    for cond in COND_ORDER:
        sub = data[data.condition.astype(str).eq(cond)].set_index('axis_id').reindex(AXIS_ORDER)
        color = CONDITION_COLORS[cond]
        for ct in ['D1', 'D2']:
            values = sub[ct].to_numpy(float)
            if not np.isfinite(values).all():
                if REQUIRE_COMPLETE_RADAR:
                    raise ValueError(f'Six-curve radar has missing data: {cond} {ct}')
                print(f'Skipping incomplete radar line: {cond}/{ct}')
                continue
            xy = vectors * (values / rmax)[:, None]
            closed_xy = np.vstack([xy, xy[:1]])
            if RADAR6_FILL_ALPHA > 0:
                ax.fill(closed_xy[:, 0], closed_xy[:, 1], color=color,
                        alpha=RADAR6_FILL_ALPHA, zorder=1)
            ax.plot(closed_xy[:, 0], closed_xy[:, 1],
                    color=color, linestyle=CELLTYPE_LINE_STYLES[ct],
                    linewidth=RADAR6_LINE_WIDTH_PT,
                    marker=CONDITION_RADAR_MARKERS[cond],
                    markersize=RADAR6_MARKER_SIZE_PT,
                    markerfacecolor='white', markeredgecolor=color,
                    markeredgewidth=0.65, zorder=3)

    if RADAR6_SHOW_LEGENDS:

        ct_handles = [Line2D([], [], color='#515151', lw=1.8, linestyle=CELLTYPE_LINE_STYLES['D1'], label='MSN-D1'),
                      Line2D([], [], color='#515151', lw=1.8, linestyle=CELLTYPE_LINE_STYLES['D2'], label='MSN-D2')]
        cond_handles = [Line2D([], [], color=CONDITION_COLORS[c],
                               ls='-', lw=1.8,
                               marker=CONDITION_RADAR_MARKERS[c], markersize=3.1,
                               markerfacecolor='white', label=c)
                        for c in COND_ORDER]

        legend_x = (RADAR6_MARGIN_LEFT_MM + RADAR6_AXES_WIDTH_MM + 5.0) / w
        legend1 = fig.legend(handles=ct_handles, loc='upper left',
                             bbox_to_anchor=(legend_x, 0.76), frameon=False,
                             title='Cell type', title_fontsize=FONT_SIZE_PT,
                             fontsize=FONT_SIZE_PT - 0.2, borderaxespad=0)
        fig.add_artist(legend1)
        fig.legend(handles=cond_handles, loc='upper left',
                   bbox_to_anchor=(legend_x, 0.51), frameon=False,
                   title='Condition', title_fontsize=FONT_SIZE_PT,
                   fontsize=FONT_SIZE_PT - 0.2, borderaxespad=0)

    ax.set_title('WT / Heter / Homo', fontsize=FONT_SIZE_PT + 1, pad=12)
    out = OUT_DIR / 'Figure4_CV_five_axes_D1_D2_radar_overlay_6_curves.pdf'
    _save_figure(fig, out)
    print('Saved:', out, ' | shared Rmax =', rmax)
    return out

radar6_pdf = make_six_curves_radar_pdf(radar_summary, radar_max_used)
